# STEP03. Q1 지원 필요도와 복지서비스 후보

## 분석 질문

- 고립·은둔 청년에게 필요한 지원에 맞는 복지서비스 후보의 존재와 영역별 구성 확인

## 입력

- 2023년 실태조사 은둔 4,448명과 고립 보조 3,032명, `data/processed/demand_2023_*.csv`와 원 설문 코드북
- SSIS 중앙 461건·지자체 4,827건, `data/processed/services_master.csv`와 메타데이터
- STEP 02 검수 배정·R1~R4 원판정·합의안, `logs/validation/q1_round1_review/`

## 출력

- `outputs/03_q1/tables/`의 수요, 기존·사람 판정 반영 공급, 순위, 정답셋 및 판정별 추적표
- `outputs/03_q1/figures/`의 기존 비교 그림과 사람 판정 반영 그림
- `outputs/03_q1/tables/access_*.csv`와 `figures/07_access_choice_reduction.png`, 신청·이용 조건별 후보
- `access_home_verification.csv`, `access_funnel_verified.csv`, `access_verification_metrics.csv`, 집 조건 후보의 원문 검증 결과
- `access_final_verification.csv`, `access_user_check.csv`, 누락 검증을 합친 최종 판정표와 사용자 확인 대상
- 원문 검증 입력 `logs/validation/q1_access_verification/`, 판정표와 검수 표본 보정 판정
- 누락 검증 입력 `nonhome_candidates_verification.csv`, 집 조건이 아닌 청년 대상 후보의 판정표
- 물리적 접근성 재검토 근거는 `logs/validation/q1_access_verification/physical_access_review.csv`
- 혼합형 9건의 세부 유형과 신청·이용 근거는 같은 판정표의 subtype 열에 수록함

## 구성

- 입력 확인 → 수요 기술 분석 → 기존 공급·순위 재현 → 사람 판정 반영·민감도 → 신청·이용 조건별 선택지 → 집 조건 후보 원문 검증 → 탐색 회귀 부록

## 수행 범위

- Q21 10개 영역의 문항별 유효 응답 기준 비가중 비율과 SSIS 등록 서비스 후보 수·순위 비교
- ⑪ 주거는 공급표 별도 표시
- 연구 범위는 Q1 및 조사·서비스 원문에서 확인 가능한 지원 내용에 한정함

## 전체 수행 흐름

- 마스터·조사 입력 및 코드북 점검
- 주 지표 매우 필요(4), 보조 지표 필요(3+4), 고립 보조 별도 집계
- 기존 S1~S3 × 영역 기준 계산 및 잠정 참조값 재현
- 1차 사람 정답셋 구성, 고정 자동 규칙 비교, Q1 분석 레이어 반영
- 직접 회복과 일반 돌봄, 경제 main과 candidate, 경계 정의에 따른 순위 비교
- 엄격 S2 후보에 신청·이용 조건을 누적 적용한 영역별·시군구별 후보 수 비교
- 엄격 S2 후보에 신청·이용 조건을 누적 적용한 영역별·시군구별 후보 수 비교

In [ ]:
import os, sys, textwrap, platform, subprocess
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display, Markdown, Image
from src.gap_index import q1_demand as D, q1_compare as C, q1_supply as S
from src.preprocess.keywords import Q21_LABELS

_fonts = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams.update({
    "font.family": ([f for f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans KR"] if f in _fonts][:1] or ["sans-serif"]),
    "axes.unicode_minus": False, "figure.dpi": 100, "savefig.dpi": 200, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#a3a29d",
    "axes.labelcolor": "#3f3e3b", "xtick.color": "#3f3e3b", "ytick.color": "#3f3e3b",
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.grid": True, "grid.color": "#ecebe7", "grid.linewidth": 0.7, "axes.axisbelow": True,
    "legend.frameon": False, "legend.fontsize": 9,
})
INK, INK2, MUTED = "#1f1f1d", "#52514e", "#8a8984"
C_HIKI, C_ISO = "#2a78d6", "#eb6834"                 # 집단: 은둔 / 고립 보조
C_CENTRAL, C_LOCAL = "#2a78d6", "#eb6834"            # 출처: 중앙 / 지자체
C_M4, C_M34 = "#1d4f91", "#86b4ec"                   # 측정: 매우 필요(4) / 필요(3+4)
BLUES = LinearSegmentedColormap.from_list("blues", ["#f4f8fd", "#86b4ec", "#1d4f91"])
DIVERGE = LinearSegmentedColormap.from_list("div", ["#c2551f", "#f6d2bd", "#f7f7f5", "#bcd6f5", "#1d4f91"])
CIRC = "①②③④⑤⑥⑦⑧⑨⑩⑪"
SHORT = {"q21_01": "머물 수 있는 공간", "q21_02": "일상생활 회복", "q21_03": "혼자 하는 취미·문화·체육", "q21_04": "같이 하는 취미·문화·체육",
         "q21_05": "전문가 상담·치료", "q21_06": "또래·멘토 상담·지지", "q21_07": "교육·학습", "q21_08": "취업(정보)·일 경험",
         "q21_09": "공동생활", "q21_10": "경제적 지원", "q21_11": "주거(조사 문항 없음)"}
AREA = {q: f"{CIRC[i]} {SHORT[q]}" for i, q in enumerate(SHORT)}
SC_KO = {"S1_A": "S1 청년 특화(A)", "S2_AB_strict": "S2 A+B(연령 근거)", "S2_AB": "S2 A+B(확장)", "S3_ABU": "S3 A+B+U(상한)"}
SC_SHORT = {"S1_A": "S1", "S2_AB_strict": "S2 strict", "S2_AB": "S2 확장", "S3_ABU": "S3"}
MEAS_KO = {"very_needed_4": "매우 필요(4)", "needed_3_4": "필요(3+4)"}

OUT = ROOT / "outputs" / "03_q1"; TBL, FIG = OUT / "tables", OUT / "figures"
TBL.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
TABLES = {}


def save_csv(df, name):
    """표시한 객체를 그대로 CSV(utf-8-sig, index 없음)로 저장한다."""
    df.to_csv(TBL / f"{name}.csv", index=False, encoding="utf-8-sig")
    TABLES[name] = df
    return df


def save_fig(fig, name, note):
    """하단에 대상·표본·분모·잠정 여부 캡션을 붙여 PNG(200dpi)로 저장하고 표시한다."""
    fig.canvas.draw()
    bb = fig.get_tightbbox(fig.canvas.get_renderer())
    fig.text(bb.x0 / fig.get_figwidth(), bb.y0 / fig.get_figheight() - 0.03, textwrap.fill(note, 125), ha="left", va="top",
             fontsize=8.5, color=MUTED, transform=fig.transFigure)
    path = FIG / f"{name}.png"
    fig.savefig(path, bbox_inches="tight", facecolor="white", pad_inches=0.25)
    plt.close(fig)
    display(Image(filename=str(path), width=980))
    return path


def show_table(df, title, fmt=None, note=None, max_rows=None):
    """읽기용 표시(원본 수치는 CSV에 따로 저장). 행이 많으면 앞부분만 보이고 전체는 CSV에 있다."""
    shown = df if max_rows is None else df.head(max_rows)
    st = shown.style.hide(axis="index").format(fmt or {}, na_rep="—")
    st = st.set_table_styles([{"selector": "th", "props": "background:#f3f2ee;color:#1f1f1d;font-weight:600;text-align:center;padding:4px 8px"},
                              {"selector": "td", "props": "padding:3px 8px"}])
    display(Markdown(f"**{title}**" + (f" (앞 {max_rows}행만 표시, 전체 {len(df)}행은 CSV)" if max_rows and len(df) > max_rows else "")))
    display(st)
    if note:
        display(Markdown(f"<span style='color:#8a8984;font-size:90%'>{note}</span>"))


def md(text):
    display(Markdown(text))


def pp(x, d=1):
    return f"{x:.{d}f}%p"


def pc(x, d=1):
    return f"{100 * x:.{d}f}%"


head = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=ROOT).stdout.strip()
import statsmodels, scipy
print(f"python {platform.python_version()} | pandas {pd.__version__} | statsmodels {statsmodels.__version__} | scipy {scipy.__version__} | git HEAD {head}")

# PART A. 입력 검증

## 1단계. 마스터·조사 파일의 형태와 메타를 점검한다
분석 전에 행 수, 코드 범위, 결측, 메타 해시, 코드북 문구를 실제로 비교한다. `fail`이 하나라도 있으면 아래 셀이 오류로 멈춘다. `note`는 실패가 아니라 해석에 영향을 주는 사실의 기록이다.

In [ ]:
hiki, iso = D.load_demand()
m = S.load_master()
val = save_csv(D.validate_inputs(m, hiki, iso), "input_validation")
cnt = val["status"].value_counts()
show_table(val[val["status"].ne("pass") & ~val["check"].str.contains("결측 수")][["check", "expected", "observed", "status", "note"]],
           "표 A-1. 입력 검증에서 pass가 아닌 항목(결측 수 기록 제외)", note="전체 항목과 문항별 결측 수는 `input_validation.csv`.")
if val["status"].eq("fail").any():
    raise RuntimeError("input_validation에 fail이 있어 중단합니다:\n" + val[val["status"].eq("fail")].to_string())

In [ ]:
miss = val[val["check"].str.contains("결측 수")]
md(f"""**관찰.** 점검 {len(val)}개 중 pass {cnt.get('pass', 0)}, note {cnt.get('note', 0)}, fail {cnt.get('fail', 0)}.
마스터 {len(m):,}행 × {m.shape[1]}열, 은둔 {len(hiki):,}명, 고립 보조 {len(iso):,}명. Q21 결측은 은둔 문항별 최대 {miss[miss['check'].str.contains('hikikomori')]['observed'].astype(int).max()}명,
고립 보조 최대 {miss[miss['check'].str.contains('isolated_aux')]['observed'].astype(int).max()}명이고, 은둔 q4 결측은 {int(hiki['q4'].isna().sum())}명이다.

**해석과 한계.** 입력 형태에 어긋남은 없어 분석을 진행한다. 다만 (1) 은둔·고립 코드북의 Q21 문구가 다르고 q4도 정의가 달라(고립 시트는 '고립 기간') 집단 직접 비교에는 한계가 있으며, q4는 은둔 안에서만 쓴다.
(2) Q1 기준 마스터와 앱 영역 v3 적용본의 열 수는 명세에 함께 기록한다. 실행에 사용한 열 수와 순서는 메타데이터로 검증한다.
(3) `meta.code_commit`과 현재 HEAD가 달라 note가 남았다. 마스터 입력 코드의 파일 해시는 일치한다.""")

# PART B. 수요 — 지원 필요도

Q21은 '(은둔/고립)생활을 벗어나기 위한 도움 필요도'를 영역 10개에 대해 1(전혀 필요하지 않음)~4(매우 필요함)로 묻는다. 모든 비율은 **문항별 유효 응답(1~4)** 을 분모로 하고 결측을 0으로 바꾸지 않는다.

### Decision Box 1 — 주 지표는 '매우 필요(4)', '필요(3+4)'는 보조
- **문제:** '필요'를 3~4로 볼지 4만으로 볼지. 정의에 따라 영역 간 변별력이 달라진다.
- **선택:** 주 지표 `very_needed_4`(응답 4), 보조 지표 `needed_3_4`(응답 3 또는 4)를 모든 표에 병기한다(`q1_demand.MEASURES`).
- **근거:** **EDA(STEP01) 결과를 본 뒤 선택했다**(사전 지정이 아님). 아래 셀에서 보듯 3+4는 대부분 영역에서 높아 영역 간 차이가 압축된다.
- **결론에 미친 영향/한계:** 주 지표에서 순위가 달라 보이는 영역도 보조 지표에서는 차이가 작을 수 있다. 어느 쪽이 '맞는' 정의인지는 이 데이터로 판단할 수 없다.
- **후속 검증:** 두 지표의 순위를 모든 비교표에 나란히 둔다(`demand_rank_4`, `demand_rank_34`).

## 1단계. 은둔 집단의 영역별 지원 필요도
**목적:** 영역별 '매우 필요'·'필요' 비율과 Wilson 95% 구간, 바로 아래 순위 영역과 구간이 겹치는지(순위 구분력)를 본다. **계산:** `q1_supply.demand_table`을 두 집단에 적용.

In [ ]:
dem = save_csv(D.demand_by_group(hiki, iso), "demand_by_area")
hk = dem[(dem["group"] == "hikikomori") & dem["primary_measure"]].sort_values(["rank", "area"])
ORDER = hk["area"].tolist()                         # 은둔 '매우 필요(4)' 수요 순위 순서(이후 모든 그림의 행 순서)
tab = hk[["rank", "area_label", "n_respondents", "missing_n", "k", "rate", "ci95_lo", "ci95_hi", "ci_overlap_next"]].copy()
tab34 = dem[(dem["group"] == "hikikomori") & dem["measure"].eq("needed_3_4")].set_index("area").loc[ORDER]
tab["rate_needed_3_4"], tab["rank_3_4"] = tab34["rate"].to_numpy(), tab34["rank"].to_numpy()
show_table(tab, "표 B-1. 은둔 집단 영역별 지원 필요도(주 지표 '매우 필요', 순위 순)",
           fmt={"rate": "{:.3f}", "ci95_lo": "{:.3f}", "ci95_hi": "{:.3f}", "rate_needed_3_4": "{:.3f}", "k": "{:,.0f}"},
           note="분모는 영역별 유효 응답자. ci_overlap_next = 바로 아래 순위 영역과 Wilson 구간이 겹침. 전체 표(고립 보조 포함)는 demand_by_area.csv.")

In [ ]:
top3 = hk.head(3)
n_over = int(hk["ci_overlap_next"].dropna().sum()); n_cmp = int(hk["ci_overlap_next"].notna().sum())
hi34 = int((tab34["rate"] >= 0.7).sum())
md(f"""**관찰.** '매우 필요' 비율이 가장 높은 세 영역은 {', '.join(f"{r.area_label}({pc(r.rate)})" for r in top3.itertuples())}이고, 가장 낮은 영역은 {hk.iloc[-1]['area_label']}({pc(hk.iloc[-1]['rate'])})이다.
바로 아래 순위와 Wilson 구간이 겹치는 경우는 {n_over}/{n_cmp}쌍이다. '필요(3+4)'는 10개 영역 중 {hi34}개에서 70% 이상이다(최저 {pc(tab34['rate'].min())}, 최고 {pc(tab34['rate'].max())}).

**해석과 한계.** ③·②·①의 점추정 순위는 우선 검토 순서를 정하는 데 참고한다. 문항별 Wilson 구간의 중첩은 같은 응답자가 답한 문항 간 차이의 검정 결과가 아니며, 순위 차이의 불확실성이 남는다. 구간은 비가중 응답자 자료에서 계산했다.""")

### Decision Box 2 — 집단 비교는 연령 직접 표준화, 표준인구는 pooled(주)와 은둔 구성(민감도)
- **문제:** 은둔과 고립 보조집단의 연령 구성이 달라 조율 차이에 연령 효과가 섞인다. 표준인구를 무엇으로 둘지.
- **선택:** 연령 3층(19-24/25-29/30-34)에 직접 표준화. 주 표준인구는 두 집단 합산 응답자 구성(`pooled`, 7,480명 고정 가중치), 민감도는 은둔 구성(`hikikomori`). 차이 = 은둔 − 고립 보조, 95% 구간은 정규근사(`q1_demand.age_standardized_comparison`).
- **근거:** 어느 한 집단의 구성을 기준으로 삼으면 그 집단에 유리하지 않게 중립 기준이 필요하다. 구간은 비확률 표본 내 표집변동만 반영한다.
- **결론에 미친 영향/한계:** 오즈비(Appendix)와 다른 지표다(%p 차이 vs 비). 두 값을 혼용하지 않는다. 연령 외 차이(추출 기준, 문항 문구)는 표준화로 보정되지 않는다.
- **후속 검증:** 두 표준인구의 차이를 아래 표 `crude_minus_std_pp`와 민감도 비교로 확인.

## 2단계. 연령 표준화 집단 차이(은둔 − 고립 보조)
**목적:** 영역별 조율 차이가 연령 구성 때문인지 본다.

In [ ]:
std = save_csv(D.age_standardized_comparison(hiki, iso), "group_comparison_age_std")
sp = std[(std["standard"] == "pooled")].copy()
sp["_o"] = sp["area"].map({a: i for i, a in enumerate(ORDER)})
sp = sp.sort_values(["measure", "_o"], ascending=[False, True])
show_table(sp[["measure", "area_label", "crude_hiki", "crude_iso", "crude_diff_pp", "std_diff_pp", "std_diff_ci95_lo_pp", "std_diff_ci95_hi_pp", "crude_minus_std_pp"]],
           "표 B-2. 은둔 − 고립 보조 차이(표준인구 pooled, %p)", fmt={c: "{:.3f}" for c in ["crude_hiki", "crude_iso"]} | {c: "{:.2f}" for c in ["crude_diff_pp", "std_diff_pp", "std_diff_ci95_lo_pp", "std_diff_ci95_hi_pp", "crude_minus_std_pp"]},
           note="구간은 정규근사, 이 표본 안의 표집변동만 반영. 은둔 구성 표준인구(민감도)는 group_comparison_age_std.csv.")

In [ ]:
v4 = sp[sp["measure"] == "very_needed_4"]
excl0 = v4[(v4["std_diff_ci95_lo_pp"] > 0) | (v4["std_diff_ci95_hi_pp"] < 0)]
hs = std[(std["standard"] == "hikikomori") & (std["measure"] == "very_needed_4")].set_index("area")
agree = int((np.sign(hs["std_diff_pp"]) == np.sign(v4.set_index("area")["std_diff_pp"].reindex(hs.index))).sum())
big = v4.loc[v4["std_diff_pp"].abs().idxmax()]
md(f"""**관찰.** '매우 필요'의 표준화 차이(은둔 − 고립 보조)는 영역별 {pp(v4['std_diff_pp'].min())} ~ {pp(v4['std_diff_pp'].max())}이고, 절대값이 가장 큰 영역은 {big['area_label']}({pp(big['std_diff_pp'])})이다.
95% 구간이 0을 포함하지 않는 영역은 {len(excl0)}개({', '.join(excl0['area_label']) if len(excl0) else '없음'})이다.
표준화 전후 차이의 최대 변화는 {pp(v4['crude_minus_std_pp'].abs().max(), 2)}이고, 은둔 구성 표준인구(민감도)에서 차이의 부호가 pooled와 같은 영역은 10개 중 {agree}개이다.

**해석과 한계.** 연령 구성을 맞췄을 때의 수치 변화가 위 범위다. 표준화 후에도 추출 기준과 문항 문구가 서로 달라, 남은 차이를 고립·은둔 상태의 차이로 분리할 수 없다. 구간은 모집단 추정이 아니다.""")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 5.6), sharey=True, gridspec_kw={"width_ratios": [1.15, 1], "wspace": 0.06})
y = np.arange(len(ORDER))[::-1]
for meas, col, mk, off in (("very_needed_4", C_M4, "o", 0.16), ("needed_3_4", C_M34, "s", -0.16)):
    d = dem[(dem["group"] == "hikikomori") & (dem["measure"] == meas)].set_index("area").loc[ORDER]
    a1.errorbar(d["rate"] * 100, y + off, xerr=[(d["rate"] - d["ci95_lo"]) * 100, (d["ci95_hi"] - d["rate"]) * 100], fmt=mk, color=col, ms=6, lw=1.4, label=MEAS_KO[meas])
    s_ = std[(std["standard"] == "pooled") & (std["measure"] == meas)].set_index("area").loc[ORDER]
    a2.errorbar(s_["std_diff_pp"], y + off, xerr=[s_["std_diff_pp"] - s_["std_diff_ci95_lo_pp"], s_["std_diff_ci95_hi_pp"] - s_["std_diff_pp"]], fmt=mk, color=col, ms=6, lw=1.4, label=MEAS_KO[meas])
a2.axvline(0, color=INK2, lw=1)
a1.set_yticks(y, [f"{i + 1}. {AREA[a]}" for i, a in enumerate(ORDER)])
a1.set_xlabel("은둔 응답자 중 비율 (%) · 점 = 비율, 선 = Wilson 95% 구간"); a2.set_xlabel("은둔 - 고립 보조, 연령 표준화 차이 (%p) · 선 = 95% 구간")
a1.set_title("은둔 청년의 영역별 지원 필요도"); a2.set_title("고립 보조집단과의 차이 (연령 표준화)")
a1.legend(loc="lower right"); a1.set_xlim(0, 100)
save_fig(fig, "01_demand_need_rates", "대상: 2023 실태조사 은둔 응답자 4,448명(왼쪽, 영역별 유효 응답 기준 n약 4,444~4,446)과 고립 보조집단 3,032명(오른쪽 비교). 비가중·비확률 표본이며 구간은 이 표본 안의 표집변동만 나타낸다. "
         "왼쪽 행 순서 = 은둔 '매우 필요' 비율 순위. 오른쪽 표준인구 = 두 집단 합산 연령 구성(pooled), 차이는 은둔 - 고립 보조. 은둔·고립 추출 기준과 Q21 문구가 달라 직접 비교에 한계가 있다.")

### Decision Box 3 — 결측은 0으로 바꾸지 않는다
- **문제:** Q21 무응답(`.`, 빈칸)과 필요 영역 수·쌍 지표의 분모.
- **선택:** 영역별 비율은 해당 문항 유효 응답(1~4)만 분모. 필요 영역 수는 **10문항이 모두 유효한 응답자만**(나머지는 제외하고 수를 기록). 쌍 지표는 **두 문항이 모두 유효한** 응답자가 분모(쌍마다 분모가 다름).
- **근거:** 결측을 0(필요 없음)으로 세면 비율이 낮아지고 필요 영역 수가 과소 집계된다. 결측은 문항당 은둔 최대 4명, 고립 최대 1명으로 적다.
- **결론에 미친 영향/한계:** 필요 영역 수는 완전 응답자 기준이라 영역별 비율과 분모가 다르다(`n_valid`, `n_excluded_missing` 열).
- **후속 검증:** 제외 인원 수를 표에 남겼다. 결측 응답이 무작위인지는 확인하지 못했다.

### Decision Box 4 — 동일 응답(straightlining) 민감도(탐색적 추가)
- **문제:** 10문항에 모두 같은 값을 고른 응답자가 '여러 영역이 동시에 필요'라는 결과를 부풀릴 수 있다.
- **선택:** 10문항이 모두 유효하고 값이 전부 같은 응답자를 `straightline`으로 정의하고, 필요 영역 수·쌍 지표를 전체(`all`)와 이들을 제외한(`excl_straightline`) 두 표본으로 병기한다.
- **근거:** 응답 성향이 복합 욕구 지표를 인위적으로 높일 수 있다. **이 민감도는 사전 계획이 아닌 탐색적 추가**이며 동일 응답이 곧 불성실 응답이라는 판단은 하지 않는다(실제로 모두 '전혀 필요하지 않음' 또는 '매우 필요함'일 수 있다).
- **결론에 미친 영향/한계:** 제외 후 결과가 거의 변하지 않으면 해당 결론은 응답 성향에 덜 민감하다고 말할 수 있을 뿐이다.
- **후속 검증:** 아래 표에서 두 표본을 비교.

## 3단계. 필요한 영역은 몇 개인가
**목적:** 한 사람이 여러 영역을 동시에 필요로 하는 정도를 본다(영역 10개 중 '매우 필요' 또는 '필요' 응답 개수). **분모:** 10문항 완전 응답자.

In [ ]:
nd = save_csv(D.need_count_distribution(hiki, iso), "need_count_distribution")
ns = save_csv(D.need_count_summary(hiki, iso), "need_count_summary")
show_table(ns, "표 B-3. 필요 영역 수 요약(10문항 완전 응답자)", fmt={c: "{:.2f}" for c in ["mean", "median", "q1", "q3"]} | {c: "{:.3f}" for c in ["share_0", "share_10", "straightline_share", "all_four_share"]},
           note="excl_straightline = 10문항 응답이 모두 같은 응답자를 제외(탐색적 민감도). straightline_share·all_four_share는 완전 응답자 중 비율.")

In [ ]:
g = lambda grp, me, sa, col: ns[(ns["group"] == grp) & (ns["measure"] == me) & (ns["sample"] == sa)][col].iloc[0]
ge5 = lambda grp, me: nd[(nd["group"] == grp) & (nd["measure"] == me) & (nd["n_areas"] == 5)]["share_at_least"].iloc[0]
md(f"""**관찰.** '매우 필요'인 영역 수의 중앙값은 은둔 {g('hikikomori','very_needed_4','all','median'):.0f}개(사분위 {g('hikikomori','very_needed_4','all','q1'):.0f}~{g('hikikomori','very_needed_4','all','q3'):.0f}), 고립 보조 {g('isolated_aux','very_needed_4','all','median'):.0f}개이고,
5개 이상인 응답자는 은둔 {pc(ge5('hikikomori','very_needed_4'))}, 고립 보조 {pc(ge5('isolated_aux','very_needed_4'))}이다. 10개 모두 '매우 필요'는 은둔 {pc(g('hikikomori','very_needed_4','all','share_10'))}, 0개는 {pc(g('hikikomori','very_needed_4','all','share_0'))}이다.
동일 응답은 은둔 완전 응답자의 {pc(g('hikikomori','very_needed_4','all','straightline_share'))}, 이 중 10문항 모두 '매우 필요'는 {pc(g('hikikomori','very_needed_4','all','all_four_share'))}(전체 대비)이다.
이를 제외하면 은둔 '매우 필요' 영역 수의 평균은 {g('hikikomori','very_needed_4','all','mean'):.2f} → {g('hikikomori','very_needed_4','excl_straightline','mean'):.2f}, 중앙값은 {g('hikikomori','very_needed_4','all','median'):.0f} → {g('hikikomori','very_needed_4','excl_straightline','median'):.0f}개로 바뀐다.
완전 응답자는 은둔 {int(nd[(nd.group=='hikikomori')&(nd.measure=='very_needed_4')]['n_valid'].iloc[0]):,}명(결측으로 {int(nd[(nd.group=='hikikomori')&(nd.measure=='very_needed_4')]['n_excluded_missing'].iloc[0])}명 제외)이다.

**해석과 한계.** 한 명이 여러 영역을 함께 필요로 하는 경향이 있어, 영역별 비율의 합은 사람 수가 아니다. 제외 전후 차이는 응답 성향에 대한 탐색적 민감도일 뿐 동일 응답의 원인을 말해주지 않는다. 열 문항에 같은 필요도를 실제로 느낀 응답자도 제외될 수 있다. 집단 간 분포 비교는 연령을 맞추지 않은 값이다.""")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.6), sharey=True)
for ax, meas in zip(axs, MEAS_KO):
    for off, (grp, col, lab) in zip((-0.2, 0.2), (("hikikomori", C_HIKI, "은둔"), ("isolated_aux", C_ISO, "고립 보조"))):
        d = nd[(nd["group"] == grp) & (nd["measure"] == meas)].sort_values("n_areas")
        ax.bar(d["n_areas"] + off, d["share"] * 100, width=0.38, color=col, label=f"{lab} (n={int(d['n_valid'].iloc[0]):,})")
    ax.set_xticks(range(11)); ax.set_xlabel("필요 영역 수 (10개 영역 중)"); ax.set_title(f"{MEAS_KO[meas]}인 영역 수의 분포"); ax.legend()
axs[0].set_ylabel("집단 내 비율 (%)")
save_fig(fig, "02_need_count_distribution", "대상: Q21 10문항에 모두 유효하게 응답한 응답자(은둔·고립 보조 각각). 결측이 있는 응답자는 제외. 비가중·비확률 표본, 연령 구성 미보정. "
         "은둔·고립 추출 기준과 Q21 문구가 달라 집단 간 차이는 연령 이외 요인과 분리되지 않는다.")

## 4단계. 함께 필요한 영역 쌍
**목적:** 영역 45쌍의 동시 필요(둘 다 '매우 필요') 비율과 연관성(lift, phi)을 본다. **분모:** 쌍마다 두 문항이 모두 유효한 응답자. lift = P(AB)/(P(A)P(B)), phi = 2×2 상관.

In [ ]:
pr = save_csv(D.pair_cooccurrence(hiki, iso), "need_pair_cooccurrence")
base = pr[(pr["group"] == "hikikomori") & (pr["measure"] == "very_needed_4")]
pa = base[base["sample"] == "all"].sort_values("phi", ascending=False)
show_table(pa[["area_a", "area_b", "label_a", "label_b", "n_valid_pair", "n_both", "joint_rate", "rate_a", "rate_b", "lift", "phi"]],
           "표 B-4. 은둔 · '매우 필요' · 동시 필요 상위 쌍(phi 순)", fmt={c: "{:.3f}" for c in ["joint_rate", "rate_a", "rate_b", "lift", "phi"]}, max_rows=10,
           note="전체 45쌍 × 집단 × 측정 × 표본(all / excl_straightline)은 need_pair_cooccurrence.csv.")

In [ ]:
pe = base[base["sample"] == "excl_straightline"].set_index(["area_a", "area_b"])
pa2 = pa.set_index(["area_a", "area_b"])
dphi = (pa2["phi"] - pe["phi"].reindex(pa2.index))
t1, b1 = pa.iloc[0], pa.iloc[-1]
md(f"""**관찰.** 은둔 '매우 필요' 기준 phi가 가장 큰 쌍은 {t1['label_a']} – {t1['label_b']}(phi {t1['phi']:.2f}, lift {t1['lift']:.2f}, 동시 {pc(t1['joint_rate'])})이고, 가장 작은 쌍은 {b1['label_a']} – {b1['label_b']}(phi {b1['phi']:.2f})이다.
45쌍 모두 phi가 양수인 쌍은 {int((pa['phi'] > 0).sum())}개, lift가 1 이상인 쌍은 {int((pa['lift'] >= 1).sum())}개이다. 동일 응답 제외 시 phi는 전체 {dphi.mean():.2f}(평균 변화량) 낮아지며, 제외 후에도 상위 쌍은 {Q21_LABELS[pe['phi'].sort_values(ascending=False).index[0][0]]} – {Q21_LABELS[pe['phi'].sort_values(ascending=False).index[0][1]]}이다.
쌍 분모(유효 응답자 수)는 {int(pa['n_valid_pair'].min()):,}~{int(pa['n_valid_pair'].max()):,}명이다.

**해석과 한계.** 연관은 한 응답자 안에서 함께 선택되는 경향이며 인과·서비스 묶음 근거가 아니다. 모든 영역에서 '필요' 응답이 전반적으로 높은 응답자(응답 성향)가 연관을 키울 수 있어 동일 응답 제외 결과를 병기했다. 제외 뒤 남은 연관에도 개인별 전반적 필요도와 문항별 응답 성향이 함께 들어갈 수 있다.""")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15.5, 7.4), gridspec_kw={"wspace": 0.12})
lab = [AREA[a] for a in S.DEMAND_AREAS]
for ax, col, title, cmap, vmax, fmt in ((axs[0], "joint_rate", "두 영역 모두 '매우 필요'인 응답자 비율 (%)", BLUES, None, "{:.0f}"),
                                       (axs[1], "phi", "영역 간 연관성 (phi 상관계수)", BLUES, None, "{:.2f}")):
    mat = np.full((10, 10), np.nan)
    for r in base[base["sample"] == "all"].itertuples():
        i, j = S.DEMAND_AREAS.index(r.area_a), S.DEMAND_AREAS.index(r.area_b)
        mat[i, j] = mat[j, i] = getattr(r, col) * (100 if col == "joint_rate" else 1)
    im = ax.imshow(np.ma.masked_invalid(mat), cmap=cmap, vmin=0, vmax=np.nanmax(mat), aspect="equal")
    for i in range(10):
        for j in range(10):
            if i != j:
                ax.text(j, i, fmt.format(mat[i, j]), ha="center", va="center", fontsize=7.5, color="white" if mat[i, j] > 0.6 * np.nanmax(mat) else INK)
    ax.set_xticks(range(10), lab, rotation=60, ha="right", fontsize=8); ax.set_yticks(range(10), lab if ax is axs[0] else [], fontsize=8)
    ax.grid(False); ax.set_title(title); fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
save_fig(fig, "03_need_pair_heatmap", "대상: 은둔 응답자, '매우 필요(4)' 기준, 동일 응답 제외 없는 전체 표본. 분모는 쌍마다 두 문항이 모두 유효한 응답자(약 4,440명 안팎, 쌍별 n은 need_pair_cooccurrence.csv). "
         "대각선은 비움. 비가중·비확률 표본이며 인과나 서비스 묶음의 근거가 아니다.")

### Decision Box 5 — 은둔 기간 q4: 은둔 시트 정의 확인, 7범주 유지, 고립 q4는 사용하지 않음
- **문제:** 은둔 기간(q4)을 어떻게 쓸지, 고립 시트의 q4를 집단 비교에 쓸 수 있는지.
- **선택:** 은둔 코드북에서 q4 문구를 확인해 '은둔 생활을 한지 총 얼마나 되었나요?'임을 점검했다(`input_validation.csv`). 7범주를 묶지 않고 그대로 쓴다. 고립 시트의 q4는 '고립 생활' 기간으로 정의가 달라 `q4_isolation`으로 이름을 바꿔 분석에 쓰지 않는다. q4 결측은 별도 수준이 아니라 제외한다.
- **근거:** 2024 은둔 정의(외출 빈도·이유)에 **기간 요건이 없으므로** 3개월 미만도 은둔 집단에 포함된다. 범주를 임의로 묶으면 결과를 보고 구간을 고르는 위험이 있다.
- **결론에 미친 영향/한계:** 범주별 표본이 작은 곳은 구간이 넓다(`n_level`, `min_age_stratum_n`). 기간과 필요도의 차이는 연령·외출 정도와 얽혀 있을 수 있다.
- **후속 검증:** 연령 표준화율을 병기(`age_std_rate`). 부록 회귀는 탐색적이며 본문에 쓰지 않는다.

## 5단계. 외출 정도(sq3)·은둔 기간(q4)별 수요
**목적:** 은둔 집단 안에서 하위집단별 필요도 차이를 본다. 비교 기준은 같은 영역의 은둔 전체 조율, 연령 표준화는 은둔 전체 연령 구성을 쓴다.

In [ ]:
sg = save_csv(D.subgroup_demand(hiki), "subgroup_demand")
v = sg[sg["measure"] == "very_needed_4"]
lv = v.assign(lv=v["variable"] + "=" + v["level"] + " " + v["level_label"])
pv = lv.pivot(index="lv", columns="area", values="rate")[S.DEMAND_AREAS] * 100
pv.columns = [SHORT[c] for c in pv.columns]
pv.insert(0, "n_level", lv.drop_duplicates("lv").set_index("lv")["n_level"])
pv = pv.reset_index().rename(columns={"lv": "하위집단"})
show_table(pv, "표 B-5. 은둔 하위집단별 '매우 필요' 비율 (%)", fmt={c: "{:.1f}" for c in pv.columns[2:]} | {"n_level": "{:,.0f}"},
           note="하위집단 변수 결측자는 제외. 구간·연령 표준화·필요(3+4)는 subgroup_demand.csv.")

In [ ]:
rng = v.groupby(["variable", "area_label"])["rate"].agg(lambda s: 100 * (s.max() - s.min()))
lo = v[v["variable"] == "q4"]
small = sg.drop_duplicates(["variable", "level"]).sort_values("n_level").iloc[0]
vs = sg[(sg["measure"] == "very_needed_4")]
wide = lambda var, col: vs[vs["variable"] == var].pivot(index="area_label", columns="level", values=col)
o, d_ = wide("sq3", "rate"), wide("q4", "rate")
o_s, d_s = wide("sq3", "age_std_rate"), wide("q4", "age_std_rate")
o_gap, d_gap = 100 * (o["1"] - o["4"]), 100 * (d_["7"] - d_["1"])
md(f"""**관찰.** 외출 정도(sq3)별로 같은 영역 안에서 '매우 필요' 비율의 최대−최소 차이가 가장 큰 영역은 {rng['sq3'].idxmax()}({rng['sq3'].max():.1f}%p), 은둔 기간(q4)별로는 {rng['q4'].idxmax()}({rng['q4'].max():.1f}%p)이다.
가장 작은 하위집단은 {small['variable']}={small['level']} {small['level_label']}(n={int(small['n_level'])})이고, 연령 층 하나의 유효 n이 가장 작은 칸은 {int(vs['min_age_stratum_n'].min())}명이다.
방향: '자기 방에서 거의 안 나옴'(sq3=1)은 '취미생활을 위해서만 외출'(sq3=4)보다 {int((o_gap > 0).sum())}/10개 영역에서 높고(차이 {o_gap.min():.1f}~{o_gap.max():.1f}%p), 연령 표준화 후에도 {int((o_s["1"] > o_s["4"]).sum())}/10개 영역에서 높다.
'10년 이상'(q4=7)은 '3개월 미만'(q4=1)보다 {int((d_gap > 0).sum())}/10개 영역에서 높고(차이 {d_gap.min():.1f}~{d_gap.max():.1f}%p), 연령 표준화 후에도 {int((d_s["7"] > d_s["1"]).sum())}/10개 영역에서 높다. 차이가 가장 큰 영역은 {d_gap.idxmax()}({d_gap.max():.1f}%p)이다.
연령 표준화 전후 비교(은둔 전체 대비 차이, 절대값 최대): 조율 {vs['diff_vs_all_pp'].abs().max():.1f}%p, 표준화 {vs['age_std_diff_vs_all_pp'].abs().max():.1f}%p이다.

**해석과 한계.** 하위집단 차이는 같은 영역 은둔 전체와의 비교이며 다중 비교 보정을 하지 않은 기술값이다. 표본이 작은 칸은 구간이 넓어 순서를 말하기 어렵다. 양끝 범주의 차이는 중간 범주를 포함한 단조 추세나 시간에 따른 개인 변화의 근거가 되지 않는다. 기간과 외출 정도가 함께 달라질 수 있어 각 요인의 효과는 분리할 수 없다.""")

# PART C. 공급 — 기존 서비스 후보 재현

- SSIS 등록 건수 기준, 청년 범위 및 경제적 지원 분류 기준별 후보 구성
- 기존 자동 라벨로 재현한 참조 결과, 사람 판정 반영 결과는 PART E 참조


### Decision Box 6 — 잠정 비교 기준

- 기존 S2_AB_strict × v4_main을 비교 기준으로 유지함
- S1은 청년 특화 A, 엄격 S2는 A와 연령 근거 B, 확장 S2는 A·B, S3는 A·B·U 포함
- 피부양자 나이에서 파생한 U는 S3 제외


### Decision Box 7 — ⑪ 주거 제외, 다중 라벨 합산 주의, v3와 v4는 ⑩ 외에도 다르다
- **문제:** 조사에는 주거 문항이 없어 ⑪은 수요와 비교할 수 없다. 한 서비스가 여러 영역에 걸치면 영역별 건수의 합은 고유 서비스 수가 아니다.
- **선택:** ⑪은 공급표(`supply_by_scenario.csv`)에 보존하되 수요 비교 순위(`rank_in_demand_areas`)에서 제외한다. 영역별 건수를 합산하지 않으며 `supply_multilabel_overlap.csv`로 합과 고유 수 차이를 확인한다. ⑩ 규칙 3종(v4_main, v4_all, v3)을 병기한다.
- **근거:** 아래 셀에서 v3와 v4 라벨 차이가 ⑩에 그치지 않음을 코드로 확인한다.
- **결론에 미친 영향/한계:** v3는 ⑩ 외에 ⑦·⑧·⑪ 라벨도 v4와 다르다. 따라서 v3 대 v4 비교를 ⑩ 정의 효과만으로 읽으면 안 된다.
- **후속 검증:** 사람 검수 후 라벨 재확인.

In [ ]:
v3m, v4m = S.area_matrix(m, "v3"), S.area_matrix(m, "v4_main")
diff = pd.DataFrame({"area": S.AREAS, "label": [SHORT[q] for q in S.AREAS], "v3_n_all_5288": [int(v3m[q].sum()) for q in S.AREAS],
                     "v4_main_n_all_5288": [int(v4m[q].sum()) for q in S.AREAS]})
diff["v4_minus_v3"] = diff["v4_main_n_all_5288"] - diff["v3_n_all_5288"]
show_table(diff, "표 C-0. 전체 5,288건 기준 영역별 건수: v3 vs v4_main(참고, 저장 안 함)", note="한 서비스가 여러 영역에 걸치므로 열의 합은 고유 서비스 수가 아니다.")
md(f"""**관찰.** ⑦ 교육·학습은 v3 {int(v3m['q21_07'].sum())}건, v4_main {int(v4m['q21_07'].sum())}건이고, ⑩은 {int(v3m['q21_10'].sum())} → {int(v4m['q21_10'].sum())}건이다. ⑧은 {int(v3m['q21_08'].sum())} → {int(v4m['q21_08'].sum())}, ⑪은 {int(v3m['q21_11'].sum())} → {int(v4m['q21_11'].sum())}건이다.
영역별 건수가 달라지는 영역은 {int((diff['v4_minus_v3'] != 0).sum())}개로, ⑩에 한정되지 않는다.""")

## 1단계. 시나리오 × ⑩ 규칙 × 영역의 서비스 건수
**목적:** 청년 범위와 ⑩ 규칙에 따라 영역별 건수가 어떻게 달라지는지 본다. 영역별 순위는 조사 문항이 있는 10개 영역 안에서만 매긴다(⑪ 제외).

In [ ]:
raw = S.supply_by_scenario(m)
sup = save_csv(C.supply_table(m, raw), "supply_by_scenario")
ref = sup[sup["is_reference"]].copy()
ref["_o"] = ref["area"].map({a: i for i, a in enumerate(ORDER + ["q21_11"])})
show_table(ref.sort_values("_o")[["area_label", "n_services", "n_central", "n_local", "n_norm_names", "fractional_count", "share_of_scenario", "rank_in_demand_areas"]],
           "표 C-1. 임시 참조값(S2_AB_strict × v4_main, provisional) 영역별 공급 후보 (수요 순위 순, ⑪ 맨 아래)",
           fmt={"fractional_count": "{:.1f}", "share_of_scenario": "{:.3f}"},
           note="n_services = 고유 서비스 수(다중 라벨이라 영역 합 ≠ 고유 서비스 수), fractional_count = 1/k 가중, n_norm_names = 지역명을 지운 서비스명 수. 전체 격자는 supply_by_scenario.csv.")

In [ ]:
nz = ref[(ref["n_services"] == 0) & ref["in_demand_comparison"]]["area_label"].tolist()
sz = sup[sup["econ_rule"] == "v4_main"].groupby("scenario")["n_services_in_scenario"].first()
tot = sup[(sup["econ_rule"] == "v4_main") & sup["in_demand_comparison"]].groupby("scenario")["n_services"].sum()
md(f"""**관찰.** 시나리오 규모는 {', '.join(f'{k} {v:,}건' for k, v in sz.items())}이다. 참조값에서 건수가 가장 많은 영역은 {ref.loc[ref['n_services'].idxmax(), 'area_label']}({int(ref['n_services'].max())}건)이고, 0건인 수요 영역은 {', '.join(nz) if nz else '없음'}이다.
⑩ 규칙을 v4_main에서 v3로 바꾸면 참조값 ⑩ 건수가 {int(sup[(sup.scenario=='S2_AB_strict')&(sup.econ_rule=='v4_main')&(sup.area=='q21_10')]['n_services'].iloc[0])}건에서 {int(sup[(sup.scenario=='S2_AB_strict')&(sup.econ_rule=='v3')&(sup.area=='q21_10')]['n_services'].iloc[0])}건이 된다.
v4_main 기준 10개 수요 영역 건수 합(고유 서비스 수 아님)은 {', '.join(f'{k} {v:,}' for k, v in tot.items())}이다.

**해석과 한계.** 0건·소수는 SSIS 수집 범위와 규칙이 함께 만든 등록 후보 건수다. 검색 표현 누락, 청년 자격 근거 부족, 실제 제공 내용의 차이를 나누어 확인해야 한다. 건수는 등록 사업 수이며 정원·예산이 아니다. 지자체 건수는 전국 합계라 한 지역 청년이 접근할 수 있는 수와 다르다.""")

## ①·② 및 경계 사례 원문 검토
보조 비교 시나리오에서 ①·② 규칙 후보는 각각 1건과 5건이다. 같은 서비스 원문을 대상으로 확장 어휘를 검색하면 미분류 ① 4건과 ② 10건이 추가로 검토 대상에 들어온다. 자유 방문이 가능한 열린 공간, 목적이 정해진 숙박·정착금, 실제 복귀 훈련, 검진의 생활습관평가는 서로 다른 사례다. 후보 소수의 원인을 구분하기 위한 점검이며, 검색 건수를 공급으로 합산하지 않는다.

청년 범위와 ⑩ 경제·⑦ 교육의 사례 검토 및 가정별 건수는 `docs/q1_label_audit.md`에 기록했다. 서비스 원문 독립 판정과 재검토도 해당 절차를 사용하며, 이 노트북은 기존 라벨로 집계한다.

2026-11-05 후속 검토는 아래 별도 보조안에서 전 행에 같은 규칙을 적용한다. 기존 본표와 공통 마스터는 보존한다.

### 후속 원문 검토와 전수 자동 재계산 — Decision Box 11

기존 사례 검토에서 정한 기준을 고정하고, 기존 검토와 중복을 제외한 무작위 40건(영역 있음 20·N 20), ①·② 검색 후보 12건, 엄격 시나리오 밖 전체 대상 표현 5건을 추가로 검토했다. 새 원문 57건이며 사람 검수나 모집단 정확도 평가가 아니다. ①·② 검색 후보 중 이전에 검토한 1건은 다시 세지 않았다. 표본 ID·원문 해시·판정·인용은 `private/q1_followup/`에 보존하고, 집계와 사례는 `docs/q1_label_audit.md`에 기록했다. 추가 외부 모델 호출은 없었다.

- **변경 이유:** 무작위 검토에서도 교육·경제 지원 누락과 주거비 오분류가 발견됐다. 응시료와 수강료, 복귀 목적과 실제 회복 지원을 구분하는 보조 규칙을 만든다.
- **적용:** 기존 5,288건 전체에 ID별 덮어쓰기 없이 같은 규칙을 적용한다. 직접 회복 지원(`revised_core`)과 일반 돌봄 확장(`revised_care`)을 분리한다. 원문 매칭 인용을 변경표에 남긴다.
- **보존:** 공식 마스터·청년 범위·v3/v4a 스냅샷은 그대로 둔다. 아래 표는 v4 기준의 보조 비교이며 기존 공식 결과를 대체하지 않는다. 새 ⑦의 수혜자가 자동 규칙으로 확인되지 않으면 N(미상)이다.
- **남은 판단:** 정책보험료·학자금 채무조정의 영역 범위, 대상과 지원내용의 불일치. 명시적 전체 대상의 B 근거는 확인했지만 청년 범위 전수 개정은 적용하지 않았다. 숫자 연령 기준 S2_AB_strict의 정의와 분모도 보존한다.
- **한계와 다음 단계:** 개발에 사용한 표본이므로 보정 후 정확도로 재사용하지 않는다. 전체 변경 후보는 추가 사람 검수를 통과한 자료가 아니다. 새 쟁점을 먼저 보고하며 HTML은 아직 작성하지 않는다.


In [ ]:
from src.gap_index import q1_followup as F
followup = F.sensitivity(m)
for name, frame in followup.items():
    save_csv(frame, name)
fs = followup['followup_supply']
followup_reference = (fs[fs.scenario.eq('S2_AB_strict') & fs.econ_rule.eq('v4_main')]
                      .pivot(index='area', columns='variant', values='n_services').reindex(S.AREAS))
show_table(followup_reference.reset_index(), '표 C-추가. 동일 청년 범위에서 영역 규칙을 바꾼 후보 수',
           note='공식 분류 / 직접 회복 중심 보조안 / 일반 돌봄 확장 보조안. 정원이나 실제 공급량이 아니다.')
fc = followup['followup_rule_changes'].groupby('variant').servId.nunique()
fr = followup['followup_rank_stability']
fr = fr[fr.scenario.eq('S2_AB_strict') & fr.econ_rule.eq('v4_main')
        & fr.supply_metric.eq('n_services') & fr.demand_measure.eq('very_needed_4')]
show_table(fr[['variant', 'spearman_rho']], '표 C-추가. 참조 칸의 수요·후보 수 순위 상관', fmt={'spearman_rho': '{:.3f}'})
md(f"직접 회복 보조안의 변경 서비스는 {fc['revised_core']}건, 돌봄 확장 보조안은 {fc['revised_care']}건이다. "
   "청년 범위와 조사 입력은 보존했다. 기준 선택에 따라 ② 후보 수와 순위 대응이 달라지므로 한 보조안을 확정값으로 고르지 않는다.")

## 2단계. 라벨 근거와 다중 라벨
**목적:** 영역 후보가 어떤 청년 범위 근거(youth_scope_basis) 위에 서 있는지, 한 서비스가 몇 개 영역에 걸치는지 본다.

In [ ]:
ev = save_csv(C.supply_evidence_review(m), "supply_evidence_review")
ov = save_csv(C.multilabel_overlap(m), "supply_multilabel_overlap")
r_ev = ev[(ev["scenario"] == "S2_AB_strict") & ev["area"].ne("q21_11")]
tot_ref = r_ev.groupby(["youth_scope", "youth_scope_basis"])["n_services"].sum().rename("건수(영역 합)").reset_index()
show_table(tot_ref, "표 C-2a. 참조값(S2_AB_strict, v4_main): 청년 범위 근거별 영역 후보 건수(10개 수요 영역 합, 중복 포함)",
           note="한 서비스가 여러 영역에 걸치면 중복 집계됨. 영역별 상세와 검수 필요 표시 수는 supply_evidence_review.csv.")
show_table(ov, "표 C-2b. 다중 라벨: 영역별 건수의 합 vs 고유 서비스 수(11개 영역 합)", fmt={"sum_over_unique": "{:.2f}", "share_multi_label": "{:.3f}"})

In [ ]:
o = ov.set_index(["scenario", "econ_rule"])
rr = o.loc[("S2_AB_strict", "v4_main")]; r3 = o.loc[("S2_AB_strict", "v3")]
rv = ev[(ev["scenario"] == "S2_AB_strict") & ev["area"].ne("q21_11")]
md(f"""**관찰.** 참조값(S2_AB_strict, v4_main)에서 시나리오 {int(rr['n_services_in_scenario'])}건 중 어느 영역에든 걸린 서비스는 {int(rr['n_with_any_area'])}건, 영역이 없는 서비스는 {int(rr['n_no_area'])}건이다.
영역별 건수의 합은 {int(rr['sum_area_counts'])}로 고유 서비스 수 {int(rr['n_unique_any_area'])}보다 크고, 여러 영역에 걸친 서비스는 {pc(rr['share_multi_label'])}이다. v3에서는 이 비중이 {pc(r3['share_multi_label'])}이다.
참조값의 10개 수요 영역 후보 중 청년 범위 근거가 B이거나 검수 필요로 표시된 건수는 각각 {int(rv[rv['youth_scope'] == 'B']['n_services'].sum())}건(영역 합), {int(rv['n_youth_scope_review'].sum())}건이다.

**해석과 한계.** 영역별 건수를 더해 '공급 총량'으로 쓰면 중복 집계가 된다. 라벨은 규칙 후보이고 검수 필요 표시가 많아 사람 검수 전이다.""")

## 3단계. 시군구 한 곳에서 접근할 수 있는 후보
**목적:** 지자체 서비스는 그 지역 주민만 쓸 수 있으므로 중앙 + 해당 시도 본청 + 해당 시군구 기준으로 접근 가능한 후보 수의 분포를 본다(참조 시나리오, 잠정).

In [ ]:
la = save_csv(C.local_access_table(m), "supply_local_access_S2_AB_strict")
la_show = la.assign(area_label=la["area"].map(lambda a: "전체 후보 수" if a == "n_total" else AREA.get(a, a)))
show_table(la_show[["area_label", "count", "min", "25%", "50%", "75%", "max", "n_regions_zero"]], "표 C-3. 시군구별 접근 가능 후보 수 분포(S2_AB_strict, v4_main, 잠정)",
           fmt={c: "{:.1f}" for c in ["min", "25%", "50%", "75%", "max"]} | {"count": "{:.0f}"},
           note="시군구 목록은 지자체 원자료에 등장한 (시도, 시군구) 쌍이며 행정구역 전체 목록이 아니다.")

In [ ]:
nt = la[la["area"] == "n_total"].iloc[0]
zr = la[la["area"].isin(S.DEMAND_AREAS)].set_index("area")["n_regions_zero"]
md(f"""**관찰.** 목록의 시군구 {int(nt['count'])}곳에서 접근 가능한 후보는 중앙값 {nt['50%']:.0f}건(최소 {nt['min']:.0f}, 최대 {nt['max']:.0f})이다. 접근 후보가 0건인 시군구 수는 영역별로 {', '.join(f"{SHORT[a]} {int(n)}곳" for a, n in zr.items() if n > 0)}이다.

**해석과 한계.** 지자체 단위 접근성은 전국 합계와 다르다. 위 분포는 규칙 후보 기준 잠정값이고, 신청 자격·연령·소득 조건은 반영하지 않았다.""")

## 4단계. ⑩ 경제적 지원 규칙과 ⑦ 교육 수혜자 민감도
**목적:** ⑩ 분류 기준(v4_main / v4_all / v3)과 ⑦에서 수혜자가 자녀·아동뿐인 서비스(dependent) 제외가 건수·순위에 미치는 영향을 본다.

In [ ]:
er = save_csv(C.econ_rule_comparison(m, raw), "econ_rule_comparison")
et = save_csv(C.econ_type_by_scenario(m), "econ_type_by_scenario")
eb = save_csv(C.edu_beneficiary_effect(m, raw), "edu_beneficiary_effect")
show_table(er[["scenario", "econ_rule", "n_q21_10", "n_central", "n_local", "share_of_scenario", "overlap_with_other_area", "rank_in_demand_areas", "delta_vs_v4_main"]],
           "표 C-4a. ⑩ 규칙별 건수(ALL = 전체 5,288건)", fmt={"share_of_scenario": "{:.3f}", "overlap_with_other_area": "{:.3f}"})
show_table(eb, "표 C-4b. ⑦ 교육·학습: 수혜자가 자녀·아동뿐인 서비스(dependent) 제외 효과", fmt={"share_dependent": "{:.3f}"},
           note="rank = 10개 수요 영역 안 건수 순위(방식 min). 규칙 후보이며 잠정.")

In [ ]:
e_ref = er[(er.scenario == "S2_AB_strict")].set_index("econ_rule")
b3 = eb.set_index("scenario").loc["S3_ABU"]; b2 = eb.set_index("scenario").loc["S2_AB_strict"]
tv = et[(et.scenario == "S2_AB_strict") & et["q21_10_tier"].isin(["main", "candidate"])].sort_values("n_services", ascending=False).head(3)
md(f"""**관찰.** 참조 시나리오에서 ⑩ 건수는 v4_main {int(e_ref.loc['v4_main','n_q21_10'])}건, v4_all {int(e_ref.loc['v4_all','n_q21_10'])}건, v3 {int(e_ref.loc['v3','n_q21_10'])}건이고, 순위는 각각 {int(e_ref.loc['v4_main','rank_in_demand_areas'])}위, {int(e_ref.loc['v4_all','rank_in_demand_areas'])}위, {int(e_ref.loc['v3','rank_in_demand_areas'])}위이다.
참조 시나리오의 main·candidate 중 ⑩ 유형 상위는 {', '.join(f"{r.q21_10_type}({r.n_services})" for r in tv.itertuples())}이다(전체 표는 econ_type_by_scenario.csv).
⑦에서 dependent를 빼면 S2_AB_strict는 {int(b2['n_q21_07_v4_main'])} → {int(b2['n_q21_07_excl_dependent'])}건(순위 {int(b2['rank_before'])} → {int(b2['rank_after'])}위), S3_ABU는 {int(b3['n_q21_07_v4_main'])} → {int(b3['n_q21_07_excl_dependent'])}건(순위 {int(b3['rank_before'])} → {int(b3['rank_after'])}위, dependent 비중 {pc(b3['share_dependent'])})이다.

**해석과 한계.** ⑩ 건수는 분류 기준에 매우 민감하다. 기준에 따라 ⑩이 늘어나는 것은 '경제적 지원 서비스가 많다'는 사실이 아니라 분류 정의의 차이다. 청년 범위가 넓어질수록(U 포함) ⑦의 자녀·아동 대상 비중이 커진다. 수혜자 판정도 규칙 후보다.""")

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(13, 9), sharey=True)
rows_y = [12 - i for i in range(10)] + [0]          # 10칸 + 간격 + ⑪(맨 아래)
areas_y = ORDER + ["q21_11"]
sv = sup[sup["econ_rule"] == "v4_main"]
for ax, sc in zip(axs.ravel(), C.SCENARIOS4):
    d = sv[sv["scenario"] == sc].set_index("area").loc[areas_y]
    yy = np.array(rows_y)
    ax.barh(yy, d["n_central"], color=C_CENTRAL, edgecolor="white", linewidth=1.5, height=0.7, label="중앙")
    ax.barh(yy, d["n_local"], left=d["n_central"], color=C_LOCAL, edgecolor="white", linewidth=1.5, height=0.7, label="지자체")
    for yv, (_, r) in zip(yy, d.iterrows()):
        ax.text(r["n_services"] + d["n_services"].max() * 0.015, yv, f"{int(r['n_services'])} (중앙 {int(r['n_central'])} · 지자체 {int(r['n_local'])})", va="center", fontsize=8, color=INK2)
    ax.axhline(1.5, color="#a3a29d", lw=0.8, ls="--")
    ax.set_yticks(yy, [AREA[a] if a != "q21_11" else "⑪ 주거 [수요 문항 없음]" for a in areas_y], fontsize=8.5)
    ax.set_xlim(0, d["n_services"].max() * 1.5); ax.set_title(f"{SC_KO[sc]}  (전체 {int(d['n_services_in_scenario'].iloc[0]):,}건)")
    ax.set_xlabel("서비스 건수 (패널별 x축 독립)")
axs[0, 0].legend(loc="lower right")
fig.suptitle("v4_main 기준 영역별 공급 후보 구성 — 잠정(provisional, 규칙 후보·사람 검수 전)", x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.97))
save_fig(fig, "04_supply_composition", "대상: SSIS 중앙 461 + 지자체 4,827건 중 시나리오별 해당 서비스. ⑩은 v4_main 기준. 영역 순서 = 은둔 '매우 필요' 수요 순위, ⑪ 주거는 조사 문항이 없어 맨 아래에 따로 표시. "
         "한 서비스가 여러 영역에 걸치므로 막대의 합은 고유 서비스 수가 아니다. 패널마다 x축 범위가 다르다. 건수는 정원·예산이 아니며 지자체는 전국 합계다.")

# PART D. 수요·공급 순위 비교

수요(은둔 응답자 비율)와 공급(서비스 건수)을 **나누지 않고** 영역별 순위만 비교한다. `rank_gap_4` = 공급 순위 − 수요 순위는 순위 차이일 뿐 부족 지표가 아니다. ⑪은 제외한다.

## 1단계. 순위 표

In [ ]:
rk = save_csv(C.demand_supply_rank(dem, sup), "demand_supply_rank")
rr_ = rk[rk["is_reference"]]
show_table(rr_[["area_label", "rate_very_needed_4", "demand_rank_4", "demand_rank_34", "n_services", "supply_rank", "rank_gap_4"]],
           "표 D-1. 임시 참조값(S2_AB_strict × v4_main, provisional): 수요 순위와 공급 순위", fmt={"rate_very_needed_4": "{:.3f}", "demand_rank_4": "{:.0f}", "demand_rank_34": "{:.0f}", "rank_gap_4": "{:+.0f}"},
           note="공급 순위 = 10개 수요 영역 안 n_services 순위(동점은 min). rank_gap_4는 순위 차이일 뿐이며 부족 지표가 아님. 16칸 전체는 demand_supply_rank.csv.")

In [ ]:
top = rr_.sort_values("demand_rank_4").head(3)
md(f"""**관찰.** 참조값에서 수요 순위 상위 3개 영역({', '.join(top['area_label'])})의 공급 순위는 각각 {', '.join(str(int(x)) for x in top['supply_rank'])}위이다.
순위 차이(공급 − 수요)의 절대값이 가장 큰 영역은 {rr_.loc[rr_['rank_gap_4'].abs().idxmax(), 'area_label']}({int(rr_.loc[rr_['rank_gap_4'].abs().idxmax(), 'rank_gap_4']):+d})이다.
공급 건수가 0인 수요 영역은 {int((rr_['n_services'] == 0).sum())}개다.

**해석과 한계.** 순위만으로 단위가 다른 두 값을 같은 척도에 놓은 것이 아니라, 높은 수요 영역에 후보가 몰려 있는지 보는 서열 비교다. 0건 영역은 순위가 동점으로 처리된다.""")

## 2단계. 12칸 격자에서 순위 모양이 유지되는가
**목적:** 시나리오 4 × ⑩ 규칙 3의 12칸(+ edu_self 4칸, 격자 밖)에서 Spearman 상관과 영역별 공급 순위의 범위를 본다. 상관은 수요 비율과 공급 값의 평균순위 상관이며 **p값을 계산하지 않는다**(n=10, 동점 많음, 탐색적 기술값).

In [ ]:
cell = save_csv(C.rank_stability_by_cell(rk), "rank_stability_by_cell")
area = save_csv(C.rank_stability_by_area(rk), "rank_stability_by_area")
cs = cell[(cell["supply_metric"] == "n_services") & (cell["demand_measure"] == "very_needed_4")]
show_table(cs[["scenario", "econ_rule", "in_grid", "spearman_rho", "n_tied_supply_values", "n_areas_zero_supply", "rho_vs_reference_supply_rank", "top3_demand_supply_ranks"]],
           "표 D-2a. 칸별 순위 상관(공급 = n_services, 수요 = 매우 필요 비율)", fmt={"spearman_rho": "{:.2f}", "rho_vs_reference_supply_rank": "{:.2f}"},
           note="top3_demand_supply_ranks = 수요 상위 3개 영역의 공급 순위(수요 순). 다른 공급 지표(n_norm_names, fractional_count)와 수요 측정(3+4)은 CSV.")
show_table(area[["demand_rank_4", "area_label", "supply_rank_min", "supply_rank_median", "supply_rank_max", "reference_supply_rank", "n_services_min", "n_services_max", "n_cells_zero", "n_cells_demand_top3_supply_bottom3", "pattern_note"]],
           "표 D-2b. 영역별 공급 순위 범위(12칸)", fmt={"supply_rank_median": "{:.1f}"})

In [ ]:
g12 = cell[cell["in_grid"] & (cell["supply_metric"] == "n_services")]
g4 = g12[g12["demand_measure"] == "very_needed_4"]
alt = cell[cell["in_grid"] & (cell["demand_measure"] == "very_needed_4")].groupby("supply_metric")["spearman_rho"].agg(["min", "max"])
wide_r = area[area["supply_rank_range"] == area["supply_rank_range"].max()]
zero_top = area.sort_values("n_cells_zero", ascending=False).iloc[0]
d4 = hk.set_index("rank")["rate"]
gap34 = 100 * (d4[3] - d4[4])
band = area[area["demand_rank_4"].isin([4, 5])]
fmt = lambda c: ", ".join(f"{r.area_label} {getattr(r, c)}칸" for r in area[area[c] > 0].itertuples()) or "없음"
md(f"""**관찰.** 12칸의 Spearman(공급 = n_services, 수요 = 매우 필요)은 {g4['spearman_rho'].min():.2f} ~ {g4['spearman_rho'].max():.2f}이고, 참조 칸은 {g4[(g4.scenario=='S2_AB_strict')&(g4.econ_rule=='v4_main')]['spearman_rho'].iloc[0]:.2f}이다.
공급 지표를 바꾸면 범위는 {', '.join(f"{k} {r['min']:.2f}~{r['max']:.2f}" for k, r in alt.iterrows())}이다. 참조 칸 대비 공급 순위 상관의 최소는 {g12['rho_vs_reference_supply_rank'].min():.2f}이다.
영역별 공급 순위의 폭이 가장 큰 영역은 {', '.join(wide_r['area_label'])}(순위 {int(area['supply_rank_range'].max())} 폭)이다. 0건인 칸이 가장 많은 영역은 {zero_top['area_label']}(12칸 중 {int(zero_top['n_cells_zero'])}칸)이고, 12칸 모두 0건인 영역은 없다.
수요 3위 이내(주 지표)이면서 공급 8위 이하인 칸은 {int(area['n_cells_demand_top3_supply_bottom3'].sum())}칸이다. 다만 수요 3·4위의 비율 차이는 {gap34:.1f}%p이고 구간이 겹쳐 이 경계는 안정적이지 않다. 보조 지표 3위 이내로 보면 {fmt('n_cells_demand34_top3_supply_bottom3')}, 주 지표 5위 이내로 보면 {fmt('n_cells_demand_top5_supply_bottom3')}이다(칸 수, 12칸 중).
주 지표 4·5위 영역의 서비스 후보는 칸에 따라 {', '.join(f"{r.area_label} {r.n_services_min}~{r.n_services_max}건" for r in band.itertuples())}이다.

**해석과 한계.** n=10이고 공급값에 동점(특히 0건)이 많아 상관은 탐색적 기술값이다. 양·음의 상관은 10개 영역의 서열이 같은 방향·반대 방향으로 움직이는 경향을 나타낸다. 자료 범위, 소수 후보의 동점, 청년 범위와 ⑩ 규칙, 중복 사업을 세는 방식에 따라 값이 달라질 수 있다. 패턴 표시(pattern_note)는 Decision Box 10의 탐색적 기준에 따른 검토 대상이며 공급 부족을 확정하지 않는다. 후보가 적은 영역은 키워드 규칙이 잡지 못했을 가능성이 있고, 사람 검수 전이라 부재와 누락을 구분할 수 없다.""")


In [ ]:
cols = [(sc, r) for sc in C.SCENARIOS4 for r in C.RULES3]
SC_TICK = {"S1_A": "S1_A", "S2_AB_strict": "S2_strict", "S2_AB": "S2_AB", "S3_ABU": "S3_ABU"}
piv = rk[rk["in_grid"]].pivot_table(index="area", columns=["scenario", "econ_rule"], values="supply_rank").loc[ORDER, cols]
dr = hk.set_index("area").loc[ORDER, "rank"]
fig, (a0, a1) = plt.subplots(1, 2, figsize=(14, 5.8), gridspec_kw={"width_ratios": [1, 12], "wspace": 0.03})
a0.imshow(np.array(dr).reshape(-1, 1), cmap=LinearSegmentedColormap.from_list("g", ["#e4e3de", "#e4e3de"]), aspect="auto")
for i, v_ in enumerate(dr):
    a0.text(0, i, str(int(v_)), ha="center", va="center", fontsize=10, color=INK, fontweight="bold")
a0.set_xticks([0], ["수요\n순위"], fontsize=9); a0.set_yticks(range(10), [AREA[a] for a in ORDER], fontsize=9); a0.grid(False)
im = a1.imshow(piv.to_numpy(), cmap=BLUES.reversed(), vmin=1, vmax=10, aspect="auto")
for i in range(10):
    for j in range(len(cols)):
        a1.text(j, i, str(int(piv.iat[i, j])), ha="center", va="center", fontsize=9.5, color="white" if piv.iat[i, j] <= 4 else INK)
a1.set_xticks(range(len(cols)), [f"{SC_TICK[s]}\n{r}" for s, r in cols], fontsize=8); a1.set_yticks([]); a1.grid(False)
for k in range(1, 4):
    a1.axvline(3 * k - 0.5, color="white", lw=3)
rj = cols.index(("S2_AB_strict", "v4_main"))
a1.add_patch(plt.Rectangle((rj - 0.5, -0.5), 1, 10, fill=False, edgecolor=INK, lw=2.6))
a1.set_title("칸(시나리오 × ⑩ 규칙)별 공급 순위 — 진할수록 건수가 많은 상위 순위")
a0.set_title("수요")
fig.colorbar(im, ax=a1, fraction=0.025, pad=0.01, label="공급 순위 (1 = 건수 최다)")
save_fig(fig, "05_rank_stability", "대상: 은둔 '매우 필요' 수요 순위 순 10개 영역 × 12칸(S1_A·S2_strict(S2_AB_strict)·S2_AB·S3_ABU × ⑩ 규칙 v4_main·v4_all·v3). 셀 = 영역별 서비스 건수(n_services) 순위(동점은 min, ⑪ 제외). "
         "굵은 테두리 = 임시 참조값(provisional, S2_AB_strict × v4_main)으로 최종 대표 시나리오가 아니다. 규칙 후보·사람 검수 전. 순위는 수요와 공급의 부족을 뜻하지 않는다.")

# PART E. 사람 정답과 수요·공급 재계산

### Decision Box 12 — 사람 판정 반영

- 서비스·문항 번호·라벨 키로 이중 검수 불일치에 합의 적용
- 청년 예의 특화·연령 구분은 기존 근거 및 검수 원문을 사용함
- 지원 영역 불명, 청년 확인필요, 신청·이용 N을 추적표와 분석 레이어에 보존함
- ⑩ 예는 생활비·반복 필수비용 main으로 집계함. 기존 candidate는 별도 대안에 사용함
- 검수자 합의 전 일치도와 고정 자동 규칙 성능을 평가함
- 개발 이력 중복 표시는 확인 가능한 자료 기준이며 독립 홀드아웃 판정 근거가 제한됨


In [ ]:
from src.gap_index import q1_final as Final

final_layers = Final.layers(m)
final_tables, gold_log = Final.analyze(m, dem, final_layers)
Final.save(final_tables, gold_log)
display(pd.Series({key: value for key, value in gold_log.items() if isinstance(value, (int, bool))}))
f1 = final_tables["human_auto_f1_summary"]
show_table(f1[f1.cohort.eq("all")].drop(columns="cohort"), "표 E-1. 자동 영역 분류 성능",
           fmt={"micro_f1": "{:.3f}", "macro_f1": "{:.3f}"},
           note="사람 불명은 라벨별 분모에서 제외. macro는 사람 양성이 있는 영역 기준")

In [ ]:
final_rank = final_tables["final_rank"]
base = final_rank.query("scenario == 'S2_AB_strict' and econ_rule == 'v4_main'")
current = base[base.variant.eq("reference_core")].set_index("area").sort_values("demand_rank_4")
display_cols = ["area_label", "rate_very_needed_4", "demand_rank_4", "n_services", "supply_rank"]
show_table(current[display_cols], "표 E-2. 사람 판정 반영 후 참조 순위",
           fmt={"rate_very_needed_4": "{:.1%}"},
           note="은둔 표본 문항별 비가중 비율과 SSIS 등록 후보 건수")
stability = final_tables["final_rank_stability"]
rs = stability.query("scenario == 'S2_AB_strict' and econ_rule == 'v4_main' and "
                     "supply_metric == 'n_services' and demand_measure == 'very_needed_4'")
econ = base[base.area.eq("q21_10")][["variant", "n_services", "supply_rank"]]
show_table(econ.merge(rs[["variant", "spearman_rho"]], on="variant"), "표 E-3. ⑩ 및 순위 민감도",
           fmt={"spearman_rho": "{:.3f}"}, note="동순위는 공동 순위, Spearman은 10개 영역의 기술값")


In [ ]:
fig, (left, right) = plt.subplots(1, 2, sharey=True, figsize=(11, 5.6), layout="constrained")
positions = np.arange(len(current))
previous = base[base.variant.eq("official")].set_index("area").reindex(current.index)
left.barh(positions, 100 * current.rate_very_needed_4, color=C_HIKI)
left.set_yticks(positions, labels=[AREA[q] for q in current.index])
left.invert_yaxis()
left.set(title="은둔 표본의 지원 필요도", xlabel="매우 필요 응답 (%)")
right.barh(positions - .18, previous.n_services, height=.34, color="#a9a8a3", label="기존 자동")
bars = right.barh(positions + .18, current.n_services, height=.34, color=C_HIKI, label="사람 판정 반영")
right.bar_label(bars, padding=3, fontsize=8)
right.margins(x=.14)
right.set(title="참조 시나리오의 영역별 공급 후보", xlabel="SSIS 등록 서비스 (건)")
right.legend(loc="lower right")
fig.suptitle("Q21 지원 필요도와 공급 후보 구성")
save_fig(fig, "06_final_demand_supply",
         "은둔 4,448명, 문항별 유효 응답·비가중. SSIS 5,288건 중 S2_AB_strict × v4_main 후보, 주거 제외.")


# PART F. 신청·이용 조건별 선택지

### Decision Box 13 — 접근 조건 누적 적용

- **문제:** 등록 후보 수만으로는 외출이 어려운 청년이 고를 수 있는 선택지 규모 확인 불가
- **선택:** 엄격 S2 후보에 `apply_barrier=0`, `use_outing=0`을 차례로 적용, 조건을 동시에 충족하는 서비스 ID 집계(`q1_access.conditions`)
- **확장안:** 신청 장벽 1(전화·우편·이메일·팩스·대리·추천·직권) 포함안 병기, 대리·추천·직권만 근거인 건은 별도 표시
- **확인 불가:** N은 접근 불가로 보지 않고 분리, N을 모두 저부담으로 가정한 상한 병기
- **근거:** 사람 검수 236건은 사람 판정, 나머지는 자동 라벨. 자동 저부담 판정의 사람 판정 일치 비율 제시
- **결론에 미친 영향/한계:** 조건 적용 후 남는 등록 건수이며 이용률·이용 가능 인원이 아님. `use_outing=0`은 급여 수령 방식 기준
- **후속 검토:** 기존 87건과 243건 판정표의 집계를 대조하고 경계 사례의 물리적 접근성을 아래에서 재판정함

In [ ]:
from src.gap_index import q1_access as Access

access = Access.analyze(m, dem, final_tables["final_supply_layer"], final_tables["human_gold"], final_layers["variants"])
Access.save(access)
fn = access["access_funnel_by_area"].set_index("area")
rows = [*current.index, "q21_11", "any_demand_area"]
fn_view = fn.loc[rows].assign(area_name=[AREA.get(q, "수요 10개 영역 고유 서비스") for q in rows])
show_table(fn_view[["area_name", "rate_very_needed_4", "n_all", "n_youth", "n_apply0", "n_home", "share_home",
                    "n_home_possible", "n_home01", "n_home_reviewed"]].rename(columns={
               "area_name": "영역", "rate_very_needed_4": "매우 필요", "n_all": "① 전체 등록", "n_youth": "② 청년 대상",
               "n_apply0": "③ 온라인·신청 불필요", "n_home": "④ 신청·이용 모두 집", "share_home": "④/②",
               "n_home_possible": "④ 상한(N 포함)", "n_home01": "④ 확장(장벽 1 포함)", "n_home_reviewed": "④ 중 사람 검수"}),
           "표 F-1. 조건 누적 적용 후 영역별 등록 후보 수(수요 순위 순)",
           fmt={"매우 필요": "{:.1%}", "④/②": "{:.1%}"},
           note="②는 엄격 S2 × v4_main. ③·④는 ② 안에서 조건을 차례로 충족한 서비스 ID 수. 영역 간 중복이 있어 합산 불가. "
                "전체 열은 access_funnel_by_area.csv, 서비스 단위 근거는 access_candidates.csv.")

In [ ]:
u = fn.loc["any_demand_area"]
big = fn[fn.in_demand_comparison & ~fn.small_denominator].sort_values("share_home")
top5 = lambda col: fn[fn.in_demand_comparison].sort_values(col, ascending=False).index[:5].tolist()
small = [AREA[q] for q in current.index if fn.at[q, "small_denominator"]]
md("\n".join([
    "**관찰**", "",
    f"- 수요 10개 영역 고유 서비스 {u.n_all:,} → {u.n_youth} → {u.n_apply0} → {u.n_home}건, 청년 대상 대비 {u.share_home:.1%}",
    *[f"- 수요 {int(fn.at[q, 'demand_rank_4'])}위 {AREA[q]} {fn.at[q, 'n_all']} → {fn.at[q, 'n_youth']} → {fn.at[q, 'n_apply0']} → {fn.at[q, 'n_home']}건"
      for q in current.index[:5]],
    f"- 청년 대상 5건 이상 영역의 ④/② 최저 {AREA[big.index[0]]} {big.share_home.iloc[0]:.1%}, 최고 {AREA[big.index[-1]]} {big.share_home.iloc[-1]:.1%}",
    f"- 청년 대상 기준과 집 조건 기준의 후보 수 상위 5개 영역 순서 {'동일' if top5('n_youth') == top5('n_home') else '변동'}",
    f"- 청년 대상 5건 미만 영역({', '.join(small)})은 비율 해석 제외", "",
    "**해석과 한계**", "",
    "- 조건을 더할수록 남는 등록 후보가 줄어듦. 이용률 감소를 뜻하지 않음",
    "- 영역 순위보다 영역별 절대 후보 수가 달라짐",
    "- ① → ② 감소에는 청년 대상 여부 판단 불가(U) 서비스의 제외가 포함됨",
]))

## 1단계. 라벨 신뢰도와 확인 불가

- 집 조건 충족 후보의 대부분이 자동 라벨
- 자동 저부담 판정과 사람 판정의 일치 정도, N 규모, 신청 장벽 1 확장안 확인

In [ ]:
rel = access["access_label_reliability"]
show_table(rel.assign(condition=rel.condition.map({"apply0": "신청 장벽 0", "outing0": "이용 외출 0", "home": "둘 다 0"})).rename(columns={
               "condition": "조건", "n_reviewed": "검수 건수", "n_auto_yes": "자동 충족", "n_human_yes": "사람 충족",
               "n_both_yes": "둘 다 충족", "n_auto_yes_human_unknown": "자동 충족·사람 N",
               "n_auto_yes_human_higher": "자동 충족·사람 1~3", "auto_yes_confirmed_share": "자동 충족 중 사람 확인",
               "human_yes_found_share": "사람 충족 중 자동 포착"}),
           "표 F-2. 자동 저부담 판정과 사람 판정(검수 236건)",
           fmt={"자동 충족 중 사람 확인": "{:.1%}", "사람 충족 중 자동 포착": "{:.1%}"},
           note="자동은 공통 마스터 라벨, 사람은 합의 반영 정답. 검수 표본은 무작위 추출이 아니어서 전체 오류율 추정에 쓰지 않음.")
unk = fn.loc[rows, ["n_youth", "n_apply_visit", "n_apply_unknown", "n_apply0_outing_required", "n_apply0_outing_unknown",
                    "n_home", "n_home_possible", "n_home01", "n_apply1_proxy_only", "n_home_no_contact_auto"]]
unk.insert(0, "영역", fn_view.area_name)
show_table(unk.rename(columns={
               "n_youth": "청년 대상", "n_apply_visit": "방문·면접 신청", "n_apply_unknown": "신청 N",
               "n_apply0_outing_required": "신청 0·이용 외출 1~3", "n_apply0_outing_unknown": "신청 0·이용 N",
               "n_home": "집 조건 충족", "n_home_possible": "상한(N 포함)", "n_home01": "확장(장벽 1 포함)",
               "n_apply1_proxy_only": "장벽 1 중 대리만", "n_home_no_contact_auto": "집 조건 중 비대면 완결(자동)"}),
           "표 F-3. 조건별 제외·확인 불가·확장안",
           note="상한 = 신청과 이용이 각각 0 또는 N. 대리만 = 자동 근거에 대리·추천·직권만 있는 건. 비대면 완결은 마스터 remote_complete=2.")

In [ ]:
h = rel.set_index("condition").loc["home"]
n = lambda col: int(h[col])
md("\n".join([
    "**관찰**", "",
    f"- 집 조건 충족 {u.n_home}건 중 사람 검수 {u.n_home_reviewed}건",
    f"- 검수 236건에서 자동 집 조건 충족 {n('n_auto_yes')}건 중 사람 확인 {n('n_both_yes')}건({h.auto_yes_confirmed_share:.1%}), 사람 N {n('n_auto_yes_human_unknown')}건, 더 높은 단계 {n('n_auto_yes_human_higher')}건",
    f"- 사람 기준 집 조건 충족 {n('n_human_yes')}건 중 자동 포착 {n('n_both_yes')}건({h.human_yes_found_share:.1%})",
    f"- N을 모두 저부담으로 가정한 상한 {u.n_home_possible}건, 신청 장벽 1 포함 확장 {u.n_home01}건",
    f"- 엄격 S2의 신청 장벽 1 중 대리·추천·직권만 근거인 건 {u.n_apply1_proxy_only}건",
    f"- 집 조건 충족 중 자동 비대면 완결 {u.n_home_no_contact_auto}건", "",
    "**해석과 한계**", "",
    "- 집 조건 후보 수는 자동 라벨 기반 근사값. 서비스 단위 판정은 2단계 원문 검증 결과 참조",
    f"- 확정 {u.n_home}건, 상한 {u.n_home_possible}건 범위에서 감소 방향 유지",
    "- 대리 신청은 본인 외출이 없을 뿐 비대면 완결이 아님. 엄격 S2에서는 건수 영향 작음",
]))

## 2단계. 집 조건 후보 원문 검증

### Decision Box 14 — 집 조건 후보 원문 검증

- **문제:** 급여 수령 기준 `use_outing=0`은 응시료·면접비·교통비·체육활동비처럼 목적 활동의 외출을 놓침
- **선택:** 집 조건 후보 87건 원문을 4개 판정으로 구분, 사람 판정 선례(시술비·관람료 면제는 외출, 월세·이사비·저축 매칭·특목고 교육비는 집)에 맞춘 기준
- **사람 검수:** 사람 판정을 사용한 초기 검증표 보존, 후속 재판정 근거는 별도 기록함
- **근거:** 판정표의 원문 근거 문구가 원문(서비스명·소개·대상·기준·지원 내용·신청 방법)에 실재하는지 코드로 확인
- **결론에 미친 영향/한계:** 87건 밖의 누락은 이 검증으로 확인 불가(아래 누락 검증에서 별도 확인), 보정 정밀도는 독립 검증 아님

In [ ]:
vh = access["access_home_verification"]
vt = access["access_funnel_verified"].set_index("area")
vm = access["access_verification_metrics"]
ACT_KO = {"exam": "응시료", "interview": "면접비", "transport": "교통비", "education": "교육비", "sports": "체육활동비",
          "photo": "사진 촬영비", "medical": "의료비", "job_search": "구직활동비", "employment": "취업·임금 사업", "": "해당 없음"}
cross = vh.pivot_table(index="verdict", columns="activity_type", values="servId", aggfunc="size", fill_value=0)
cross = cross.reindex(index=list(Access.VERDICTS)).fillna(0).astype(int)
cross = cross[[c for c in ACT_KO if c in cross.columns]]
cross.insert(0, "합계", cross.sum(axis=1))
cross.index = [Access.VERDICTS[k] for k in cross.index]
cross.columns = ["합계"] + [ACT_KO[c] for c in cross.columns[1:]]
show_table(cross.rename_axis("판정").reset_index(), "표 F-8. 집 조건 후보 원문 판정 × 활동 유형",
           note="집 조건(자동) 후보 전체의 판정. 활동 유형은 현금 수령 뒤 목적 활동이 있는 경우와 라벨 오류 유형. 전체 근거는 access_home_verification.csv.")
# 근거 문구가 원문에 실재하는지 확인
RAW = ["servNm", "summary_text_raw", "target_text_raw", "criteria_text_raw", "benefit_text_raw", "apply_text_raw", "overview_text_raw"]
squash = lambda s: "".join(str(s).split())
raw = m.set_index("servId")[RAW]
ev_found = pd.Series({r.servId: squash(r.evidence_text) in squash(" ".join(raw.loc[r.servId].fillna(""))) for r in vh.itertuples()})
ev_n, ev_ok = len(ev_found), int(ev_found.sum())
ev_missing = ev_found[~ev_found].index.tolist()
assert ev_ok == ev_n, ev_missing

In [ ]:
AREA_VIEW = {**AREA, "any_demand_area": "수요 10개 영역 고유 서비스"}
rows_v = [*current.index, "any_demand_area"]
vt_view = vt.loc[rows_v]
vt_view = pd.DataFrame({
    "영역": [AREA_VIEW[q] for q in rows_v], "청년 대상": vt_view.n_youth, "집 조건(자동)": vt_view.n_home_auto,
    "집 확인": vt_view.n_home_confirmed, "현금·외출 활동": vt_view.n_cash_outing_activity, "불명확": vt_view.n_unclear,
    "라벨 오류": vt_view.n_auto_label_error, "자동 비율": vt_view.share_home_auto, "검증 비율": vt_view.share_home_verified,
    "검증 상한 비율": vt_view.share_home_verified_upper})
show_table(vt_view, "표 F-9. 원문 검증 전후 영역별 집 조건 후보(수요 순위 순)",
           fmt={c: "{:.1%}" for c in ["자동 비율", "검증 비율", "검증 상한 비율"]},
           note="비율 분모는 청년 대상 후보(엄격 S2 × v4_main). 검증 비율은 집 확인만, 검증 상한 비율은 불명확 사례가 모두 충족한다고 가정. 5건 미만 영역은 해석 제외. 영역 간 중복이 있어 합산 불가.")
rule_ko = {"automatic": "자동 라벨", "automatic_plus_source_rule": "자동 라벨 + 원문 기준(보정)"}
show_table(pd.DataFrame({
    "규칙": vm.rule.map(rule_ko), "검수 건수": vm.n_reviewed, "집 조건 예측": vm.n_predicted_home, "사람 기준 집 조건": vm.n_human_home,
    "일치": vm.n_true_positive, "정밀도": vm.precision, "검수 표본 내 재현율": vm.recall_within_reviewed,
    "독립 검증": vm.independent.map({True: "예", False: "아니오(보정)"})}),
    "표 F-10. 검수 236건에서 집 조건 분류의 정밀도와 재현율",
    fmt={"정밀도": "{:.1%}", "검수 표본 내 재현율": "{:.1%}"},
    note="둘째 행은 사람 판정을 본 뒤 맞춘 보정 결과이며 독립 검증이 아님. 재현율은 비무작위 검수 표본 안의 값이며 전체 재현율이 아님. 원문 기준은 후보를 제거만 할 수 있어 재현율이 늘지 않음.")

In [ ]:
a = vt.loc["any_demand_area"]
n_cash = vh[vh.verdict.eq("cash_outing_activity")].activity_type.map(ACT_KO).value_counts()
n_err = vh[vh.verdict.eq("auto_label_error")].activity_type.map(ACT_KO).value_counts()
big_v = vt[vt.index.isin(S.DEMAND_AREAS) & ~vt.small_denominator].sort_values("share_home_verified")
vm_i = vm.set_index("rule")
am = vm_i.loc["automatic"]
human_kept = vh[vh.human_label_retained]
md("\n".join([
    "**관찰**", "",
    f"- 집 조건(자동) 후보 {a.n_home_auto}건 → 집 확인 {a.n_home_confirmed}건, 현금 수령은 집·목적 활동은 외출 {a.n_cash_outing_activity}건, 불명확 {a.n_unclear}건, 자동 라벨 오류 {a.n_auto_label_error}건",
    f"- 현금·외출 활동 유형: {', '.join(f'{k} {v}건' for k, v in n_cash.items())}",
    f"- 자동 라벨 오류 유형: {', '.join(f'{k} {v}건' for k, v in n_err.items())}",
    f"- 청년 대상 {a.n_youth}건 대비 자동 {a.share_home_auto:.1%} → 원문 검증 후 {a.share_home_verified:.1%}, 불명확 포함 상한 {a.share_home_verified_upper:.1%}",
    f"- 청년 대상 5건 이상 영역의 검증 비율 최저 {AREA[big_v.index[0]]} {big_v.share_home_verified.iloc[0]:.1%}, 최고 {AREA[big_v.index[-1]]} {big_v.share_home_verified.iloc[-1]:.1%}",
    f"- 사람 검수 {len(human_kept)}건은 사람 판정 유지, 검증 판정 모두 집 확인 {'일치' if human_kept.verdict.eq('home_confirmed').all() else '불일치'}",
    f"- 검수 {int(am.n_reviewed)}건에서 자동 집 조건 정밀도 {am.precision:.1%}({int(am.n_true_positive)}/{int(am.n_predicted_home)}), 검수 표본 내 재현율 {am.recall_within_reviewed:.1%}({int(am.n_true_positive)}/{int(am.n_human_home)})",
    f"- 판정표 근거 문구가 원문에 실재하는 건 {ev_ok}/{ev_n}건", "",
    "**해석과 한계**", "",
    "- 자동 집 조건 후보는 목적 활동의 외출과 취업·임금 사업을 포함해 과대 추정되었음",
    "- 원문 재판정과 규칙 검토는 AI와 기존 사람 판정 활용, 독립 사람 검수 미실시",
    "- 재현율은 비무작위 검수 표본 안의 값, 전체 재현율이 아님. 원문 기준은 후보를 제거만 하므로 값이 늘지 않음",
    f"- {a.n_home_auto}건 밖의 누락은 이 단계에서 확인 불가, 아래 누락 검증에서 확인",
    "- 검증은 원문 문구 기준이며 개인 자격·실제 이용 경험은 반영하지 않음",
]))

### 3단계. 물리적 접근성 최종 재판정

- 엄격 기준은 본인 온라인 신청 또는 별도 신청 불필요, 지원 수령·이용까지 외출 불필요
- 확장 기준은 전화·우편·팩스·이메일·대리·추천·직권 신청을 추가함. 전화 상담은 이용 방식으로 구분함
- 재학·재직·창업은 자격 요건으로 별도 기록함. 지급에 필수인 대면 교육·현장실습 등은 이용 조건에 반영함
- 혼합 사업은 세부 유형의 외출 필요성을 기록하고 전체 판정이 불명확하면 확인 불가에 포함함
- 신규 지원 중단은 운영 상태에 기록함. 신규 신청 경로가 없는 경우 물리적 접근성 미평가, SSIS 등록 분모에 포함함
- 공개 자료 추가 대조는 기존 확인 불가 24건 대상, 정부24는 같은 사업의 상세 신청·지급 절차 확인에 활용함
- 운영 상태는 공식 근거가 있는 회차 기준으로 기록함. 현재 신청 가능 건수는 집계 대상에서 제외함
- 기존 사람 판정과 원문 검증표를 보존하고 경계 사례 19건과 기존 표의 같은 기준 불일치를 별도 기록함

In [ ]:
nh_in = Access.read_nonhome_verification()
fv = access["access_final_verification"]
METHOD_KO = {"human_reuse": "사람 판정 재사용", "rule_visit_only": "규칙(방문 신청만 안내)", "source_review": "원문 판정", "official_source": "공식 자료 확인"}
REASON_KO = {"visit_only_apply": "방문 신청만 안내", "activity_outing": "목적 활동에 외출 필요", "human_label": "사람 판정 유지",
             "employment": "취업·임금 사업", "program_or_facility": "기관 프로그램·시설 이용", "local_currency_or_card": "지역화폐·카드 사용처 방문"}
cross_nh = pd.crosstab(nh_in.review_method, nh_in.verdict).reindex(index=list(METHOD_KO), columns=list(Access.NONHOME_VERDICTS), fill_value=0)
cross_nh.insert(0, "합계", cross_nh.sum(axis=1))
cross_nh.loc["합계"] = cross_nh.sum()
cross_nh.index = [METHOD_KO.get(k, k) for k in cross_nh.index]
cross_nh.columns = ["합계"] + [Access.NONHOME_VERDICTS[k] for k in cross_nh.columns[1:]]
show_table(cross_nh.rename_axis("검토 방법").reset_index(), "표 F-11. 기존 추가 검토 방법 × 판정",
           note="집 조건이 아닌 청년 대상 후보(수요 10개 영역) 전체. 초기 판정은 nonhome_candidates_verification.csv.")
nh_reason = nh_in[nh_in.verdict.eq("not_home")].reason.map(lambda r: REASON_KO.get(r, "기타")).value_counts()
show_table(nh_reason.rename_axis("집 조건 미충족 사유").reset_index(name="건수"), "표 F-12. 집 조건 미충족 확인 사유",
           note="사유는 판정표 reason 열. 한 서비스에 하나의 사유만 기록.")

In [ ]:
AREA_SHORT = lambda s: " · ".join(AREA[q] for q in s.split("|"))
uc = access["access_user_check"]
show_table(uc.assign(areas=uc.areas.map(AREA_SHORT))[
    ["servNm", "areas", "previous_verdict_label", "verdict_label", "apply_method", "support_delivery",
     "eligibility_requirements", "outing_required", "review_reason", "operation_status_label"]
].rename(columns={"servNm": "서비스", "areas": "영역", "previous_verdict_label": "기존 판정",
                  "verdict_label": "재판정", "apply_method": "신청", "support_delivery": "수령·이용",
                  "eligibility_requirements": "자격 요건", "outing_required": "외출", "review_reason": "이유",
                  "operation_status_label": "운영 상태"}),
    "표 F-13. 경계 사례 19건 재판정", note="세부 근거와 출처는 access_user_check.csv.")

In [ ]:
rows_f = [*current.index, "any_demand_area"]
vf = vt.loc[rows_f]
show_table(pd.DataFrame({
    "영역": [AREA_VIEW[q] for q in rows_f], "청년 대상": vf.n_youth, "자동 집 후보": vf.n_home_auto,
    "엄격 충족": vf.n_home_final, "엄격 미충족": vf.n_not_home_final, "확인 불가": vf.n_unclear_final,
    "미평가": vf.n_not_assessed_final,
    "엄격 비율": vf.share_home_final, "가정 상한": vf.n_home_final_upper,
    "확장 추가": vf.n_extension_addition, "확장 합계": vf.n_extension_final}),
    "표 F-14. 물리적 접근성 최종 판정(수요 순위 순)", fmt={"엄격 비율": "{:.1%}"},
    note="상한은 확인 불가 전체가 충족한다고 가정. 전체는 고유 ID 수, 영역별 행에는 중복 서비스 포함.")
ext = fv.loc[fv.extension_addition].copy()
ext["reason"] = ext.review_reason.where(ext.review_reason.ne(""), ext.verify_note)
show_table(ext[["servNm", "reason"]].rename(columns={"servNm": "서비스", "reason": "추가 이유"}),
           "표 F-15. 확장 기준에서 추가되는 서비스",
           note="최종 엄격 기준에 신청 장벽 1과 집 이용이 확인된 서비스만 추가함.")

In [ ]:
d = fn.loc[current.index].join(vt[["n_home_final", "share_home_auto", "share_home_final", "share_home_final_upper"]])
fig, (left, right) = plt.subplots(1, 2, sharey=True, figsize=(12.5, 6), gridspec_kw={"width_ratios": [1.45, 1], "wspace": 0.05})
positions = np.arange(len(d))
C_STAGE = {"n_youth": "#d6e4f5", "n_apply0": "#86b4ec", "n_home": "#2a78d6", "n_home_final": "#0f2543"}
for col, label in {"n_youth": "청년 대상(엄격 S2)", "n_apply0": "→ 온라인 신청·신청 불필요", "n_home": "→ 집 조건(자동)"}.items():
    left.barh(positions, d[col], height=.62, color=C_STAGE[col], label=label)
# 최종 값은 누락 추가로 자동 값보다 클 수 있어 겹치지 않도록 가는 막대로 표시
left.barh(positions, d.n_home_final, height=.26, color=C_STAGE["n_home_final"], label="최종 집(검증·누락 반영)")
for y, r in zip(positions, d.itertuples()):
    left.text(max(r.n_youth, 1) * 1.08 + 1.5, y, f"{r.n_youth} → {r.n_apply0} → {r.n_home} → {r.n_home_final}", va="center", fontsize=9, color=INK)
left.set_xscale("function", functions=(np.sqrt, np.square))
left.set_xlim(0, d.n_youth.max() * 2.3); left.set_xticks([0, 10, 50, 100, 200])
left.set_yticks(positions, labels=[f"{int(r.demand_rank_4)}. {AREA[q]} ({r.rate_very_needed_4:.0%})" for q, r in d.iterrows()])
left.invert_yaxis()
left.text(0.99, 0.02, "youth → apply0 → home(auto) → home(final)", transform=left.transAxes, ha="right", va="bottom", fontsize=9, color=MUTED)
left.set(title="조건을 더하고 원문 검증·누락 검증을 거친 뒤 남는 후보", xlabel="SSIS 등록 서비스 (건, 제곱근 축)")
left.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
ok = ~d.small_denominator.to_numpy()
right.barh(positions[ok] - .17, 100 * d.share_home_auto[ok], height=.32, color=C_STAGE["n_home"], label="집 조건(자동) 비율")
right.barh(positions[ok] + .17, 100 * d.share_home_final[ok], height=.32, color=C_STAGE["n_home_final"], label="최종 비율(검증·누락 반영)")
right.scatter(100 * d.share_home_final_upper[ok], positions[ok] + .17, marker="D", s=26, color=C_ISO, zorder=3, label="가정 상한(확인 불가 전부 충족)")
for y, r in zip(positions, d.itertuples()):
    text = f"{r.share_home_auto:.0%} → {r.share_home_final:.0%}" if not r.small_denominator else "청년 대상 5건 미만"
    right.text((100 * max(r.share_home_auto, r.share_home_final_upper) + 2) if not r.small_denominator else 2, y, text,
               va="center", fontsize=9, color=INK if not r.small_denominator else MUTED)
right.set_xlim(0, 100 * max(d.share_home_auto[ok].max(), d.share_home_final_upper[ok].max()) * 1.5)
right.set(title="청년 대상 후보 중 집에서 신청·이용 가능 비율", xlabel="청년 대상 후보 대비 (%)")
right.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=1)
fig.suptitle("집에서 신청하고 지원을 받을 수 있는 등록 서비스", x=0.02, ha="left", fontweight="bold")
save_fig(fig, "07_access_choice_reduction",
         f"수요 10개 영역 고유 등록 서비스 {vf.at['any_demand_area', 'n_youth']}건, 엄격 기준 충족 {vf.at['any_demand_area', 'n_home_final']}건. "
         "영역 간 중복 포함. 행 라벨은 은둔 표본의 매우 필요 순위·비율. 상한은 확인 불가 전부 충족 가정. 개인 자격과 실제 이용률은 미반영.")

In [ ]:
z = vt.loc["any_demand_area"]
md("\n".join([
    f"- 기존 집 후보 {z.n_home_auto}건 중 유지 {z.n_home_confirmed}건, 엄격 미충족 {z.n_not_home_auto_review}건, 확인 불가 {z.n_unclear}건",
    f"- 기존 집 외 후보 {z.n_nonhome}건 중 누락 발견 {z.n_missed_home}건, 미충족 {z.n_not_home_nonhome}건, 확인 불가 {z.n_unclear_nonhome}건, 확장 후보 {z.n_extension_apply1_home}건",
    f"- 최종 엄격 충족 {z.n_home_final}/{z.n_youth}건({z.share_home_final:.1%}), 미충족 {z.n_not_home_final}건, 확인 불가 {z.n_unclear_final}건, 미평가 {z.n_not_assessed_final}건",
    f"- 고용24에서 확인한 청년내일채움공제 2024년 신규 지원 중단 {z.n_new_support_discontinued}건, 기존 가입자 온라인 만기금 신청 안내 유지",
    f"- 최종 충족의 구성은 기존 집 후보에서 {z.n_home_final_from_auto_home}건, 기존 집 외 후보에서 {z.n_home_final_from_auto_not_home}건",
    f"- 확장 추가 {z.n_extension_addition}건, 확장 합계 {z.n_extension_final}건({z.share_extension_final:.1%})",
    f"- 확인 불가 {z.n_unclear_final}건이 모두 엄격 기준에 해당한다고 가정한 상한 {z.n_home_final_upper}건({z.share_home_final_upper:.1%})",
    "- 등록 자료의 0건은 SSIS에 포착된 후보가 0건임을 의미함",
    "- 2026-11-12 기준 확인한 운영 상태·근거는 최종 판정표 operation_status·operation_note·operation_source 열에 수록함",
    "- AI 원문 재판정과 규칙 검토에 기존 사람 판정 활용, 독립 사람 검수 미실시",
]))

## 4단계. 시군구 한 곳에서 보이는 선택지

- 지자체 서비스는 해당 지역 주민만 이용 가능
- 중앙 + 해당 시도 본청 + 해당 시군구 기준으로 조건별 후보 수 분포 확인

In [ ]:
rg = access["access_regional"]
rg_view = rg.pivot(index="area", columns="stage", values=["median", "n_regions_zero"]).loc[[*current.index, "q21_11"]]
rg_view.columns = [f"{'중앙값' if a == 'median' else '0건 시군구'}·{Access.STAGES[b]}" for a, b in rg_view.columns]
rg_view = rg_view[[c for s in ["youth", "apply0", "home"] for c in rg_view.columns if c.endswith(Access.STAGES[s])]]
rg_view.insert(0, "영역", [AREA[q] for q in rg_view.index])
show_table(rg_view, "표 F-4. 시군구별 조건 충족 후보 수(252곳)", fmt={c: "{:.0f}" for c in rg_view.columns[1:]},
           note="시군구 목록은 지자체 원자료에 등장한 (시도, 시군구) 쌍. 전체 분포는 access_regional.csv.")
med = rg[rg.stage.eq("home")].set_index("area")
rf = {s: rg[rg.stage.eq(s)].set_index("area") for s in ["home", "home_final", "home_final_upper"]}
show_table(pd.DataFrame({"영역": [AREA[q] for q in current.index],
                         "자동 집 조건 중앙값": [rf["home"].at[q, "median"] for q in current.index],
                         "최종 집 중앙값": [rf["home_final"].at[q, "median"] for q in current.index],
                         "최종 상한 중앙값": [rf["home_final_upper"].at[q, "median"] for q in current.index],
                         "0건 시군구(자동)": [rf["home"].at[q, "n_regions_zero"] for q in current.index],
                         "0건 시군구(최종)": [rf["home_final"].at[q, "n_regions_zero"] for q in current.index]}),
           "표 F-16. 시군구별 집 조건 후보 수: 자동 대비 최종",
           fmt={c: "{:.0f}" for c in ["자동 집 조건 중앙값", "최종 집 중앙값", "최종 상한 중앙값"]},
           note="최종 집은 원문 검증·누락 검증 반영, 상한은 확인 불가 전부 충족 가정. 수요 10개 영역만 계산. 같은 도달 범위(중앙 + 시도 본청 + 시군구).")
fin_med = rf["home_final"]["median"].loc[S.DEMAND_AREAS]
edu = fv.loc[fv.areas.str.split("|", regex=False).map(lambda areas: "q21_07" in areas)].copy()
central_ids = set(m.loc[m.source.eq("central"), "servId"])
edu_central = edu.loc[edu.servId.isin(central_ids)]
edu_added = edu_central.loc[edu_central.final_home & edu_central.source_set.eq("auto_not_home"), "servNm"]
edu_removed = edu_central.loc[~edu_central.final_home & edu_central.source_set.eq("auto_home"), "servNm"]
up_med = rf["home_final_upper"]["median"].loc[S.DEMAND_AREAS]
md("\n".join([
    "**관찰**", "",
    *[f"- {AREA[q]} 최종 집 후보 중앙값 {rf['home_final'].at[q, 'median']:.0f}건(자동 {med.at[q, 'median']:.0f}건), 0건 시군구 {rf['home_final'].at[q, 'n_regions_zero']}곳" for q in current.index[:7]],
    f"- 10개 영역 최종 집 중앙값 {fin_med.min():.0f}~{fin_med.max():.0f}건, 상한 {up_med.min():.0f}~{up_med.max():.0f}건(자동 {med.loc[S.DEMAND_AREAS, 'median'].min():.0f}~{med.loc[S.DEMAND_AREAS, 'median'].max():.0f}건)", "",
    "**해석과 한계**", "",
    f"- 교육·학습 중앙 서비스 집 후보 {edu_central.source_set.eq('auto_home').sum()}건 → {edu_central.final_home.sum()}건, 모든 252개 지역에 공통 반영",
    f"- 중앙 교육 후보 추가 {len(edu_added)}건 · {' / '.join(edu_added)}",
    f"- 중앙 교육 후보 제외 또는 확인 불가 {len(edu_removed)}건" + (f" · {' / '.join(edu_removed)}" if len(edu_removed) else ""),
    "- 시도 본청 서비스는 해당 시도 전체, 시군구 사업은 해당 시군구에만 더해 중앙값 계산",
    "- 소득·장애·재학 등 개인 자격 조건은 미반영, 실제 선택지는 더 적을 수 있음",
]))

## 5단계. 일상생활 회복의 직접 회복과 간접 지원

- 엄격 S2의 ② 후보: 직접 회복 3건, 일반 돌봄 포함 18건, 돌봄 어휘 확장 25건
- 확장으로 추가되는 서비스의 대상과 지원 내용 확인

In [ ]:
rl = access["access_recovery_layers"]
LAYER_KO = {"direct_recovery": "직접 회복", "general_care": "일반 돌봄 추가", "care_vocabulary": "돌봄 어휘 추가"}
rl_sum = rl.groupby("layer", sort=False).agg(
    n=("servId", "size"), n_reviewed=("q1_human_reviewed", "sum"),
    n_human_no=("h_q21_02", lambda s: int(s.eq("아니오").sum())),
    n_disability=("target_condition", lambda s: int(s.fillna("").str.contains("장애").sum())),
    n_home=("stage_reached", lambda s: int(s.eq("home").sum()))).reset_index()
rl_sum["cumulative"] = rl_sum.n.cumsum()
show_table(rl_sum.assign(layer=rl_sum.layer.map(LAYER_KO)).rename(columns={
               "layer": "구분", "n": "추가 건수", "cumulative": "누적", "n_reviewed": "사람 검수", "n_human_no": "사람 판정 ② 아니오",
               "n_disability": "장애 대상 조건(자동)", "n_home": "집 조건 충족"}),
           "표 F-5. ② 일상생활 회복 후보의 정의별 구성(엄격 S2)")
show_table(rl.assign(layer=rl.layer.map(LAYER_KO), benefit=rl.benefit_text_raw.fillna("").str.slice(0, 70))[
               ["layer", "servNm", "region_sido", "target_condition", "matched_term", "apply_barrier", "use_outing", "benefit"]]
           .rename(columns={"layer": "구분", "servNm": "서비스", "region_sido": "지역", "target_condition": "대상 조건(자동)",
                            "matched_term": "매칭 어휘", "apply_barrier": "신청", "use_outing": "이용", "benefit": "지원 내용 원문(앞 70자)"}),
           "표 F-6. ② 후보 서비스 목록", note="지역 빈칸은 중앙 서비스. 원문 전체와 대상 원문은 access_recovery_layers.csv.")
g = rl_sum.set_index("layer")
direct = rl[rl.layer.eq("direct_recovery")]
added = g.loc[["general_care", "care_vocabulary"]]
md("\n".join([
    "**관찰**", "",
    f"- 직접 회복 {len(direct)}건 중 서비스명에 고립·은둔을 명시한 사업 {int(direct.servNm.str.contains('고립|은둔').sum())}건",
    f"- 일반 돌봄 추가 {g.at['general_care', 'n']}건 중 장애 대상 조건 {g.at['general_care', 'n_disability']}건, 활동지원·통합돌봄·가사 간병 중심",
    f"- 돌봄 어휘 추가 {g.at['care_vocabulary', 'n']}건은 급식·숙식·가사활동 문구 매칭분",
    f"- 추가분 중 사람 검수 {added.n_reviewed.sum()}건, 그중 ② 아니오 판정 {added.n_human_no.sum()}건",
    f"- 직접 회복 {len(direct)}건 중 집 조건 충족 {g.at['direct_recovery', 'n_home']}건, 이용 방식 N {int(direct.use_outing.eq('N').sum())}건", "",
    "**해석과 한계**", "",
    f"- {' → '.join(map(str, rl_sum.cumulative))}건의 증가는 장애·질병 등 자격 조건이 있는 돌봄과 어휘 매칭분, 은둔 청년의 생활 회복을 직접 지원하는 후보 증가가 아님",
    "- 직접 회복 후보 소수는 분류 기준을 넓혀도 유지됨",
    "- SSIS 미등록 지자체·민간 사업과 청년 범위 U의 누락 가능성이 있어 실제 공급 부재로 단정 불가",
]))

## 6단계. 영역별 원문 사례

- 조건 통과, 신청 단계 제외, 이용 단계 제외, 확인 불가 사례를 고정 ID로 제시

In [ ]:
ex = access["access_examples"]
show_table(ex.assign(area=ex.area.map(AREA), stage=ex.stage_reached.map({"youth": "② 청년 대상", "apply0": "③ 신청 조건", "home": "④ 집 조건"}),
                     benefit=ex.benefit_source_text.fillna("").str.replace(r"\\n|\s+", " ", regex=True).str.slice(0, 90),
                     apply=ex.apply_source_text.fillna("").str.replace(r"\\n|\s+", " ", regex=True).str.slice(0, 70))[
               ["area", "servNm", "region", "label_source", "apply_barrier", "use_outing", "stage", "benefit", "apply", "reading"]]
           .rename(columns={"area": "영역", "servNm": "서비스", "region": "지역", "label_source": "라벨", "apply_barrier": "신청",
                            "use_outing": "이용", "stage": "도달 조건", "benefit": "지원 내용 원문", "apply": "신청 방법 원문", "reading": "해석"}),
           "표 F-7. 영역별 원문 사례", note="원문 전체는 access_examples.csv. 사례는 설명용 고정 ID이며 대표 표본이 아님.")
md("\n".join([
    "**해석과 한계**", "",
    "- 같은 현금 급여도 신청 경로에 따라 조건 충족 여부가 갈림",
    "- 수강료·활동비 환급형은 2단계 원문 검증에서 현금 수령은 집, 목적 활동은 외출로 분리",
    "- 고립·은둔 명시 사업은 이용 방식이 원문에 없어 확인 불가로 남음",
]))

## 종합 해석과 남은 경계

In [ ]:
home_med = rf["home_final"]["median"].loc[S.DEMAND_AREAS]
va = vt.loc["any_demand_area"]
same_order = vt.loc[S.DEMAND_AREAS, "rank_home_auto"].eq(vt.loc[S.DEMAND_AREAS, "rank_home_final"]).all()
top_need = ", ".join(f"{AREA[q]} {fn.at[q, 'rate_very_needed_4']:.1%}" for q in current.index[:5])
few = ", ".join(f"{AREA[q]} {fn.at[q, 'n_youth']}건" for q in current.index[:5] if fn.at[q, "small_denominator"])
md("\n".join([
    f"- **어떤 지원이 필요한가:** 은둔 표본의 매우 필요 상위는 {top_need} 순",
    f"- **어떤 서비스가 존재하는가:** 수요 상위 영역에도 청년 대상 등록 후보가 확인됨. 다만 {few}으로 영역 간 차이가 큼",
    f"- **접근 조건에 따라 선택지가 어떻게 달라지는가:** 청년 대상 {va.n_youth}건 → 자동 집 조건 {va.n_home_auto}건({va.share_home_auto:.1%}) → "
    f"최종 {va.n_home_final}건({va.share_home_final:.1%}, 기존 집 외 후보에서 {va.n_home_final_from_auto_not_home}건 추가), "
    f"확인 불가 {va.n_unclear_final}건이 모두 충족한다고 가정한 상한 {va.n_home_final_upper}건({va.share_home_final_upper:.1%}), 신청 장벽 1 확장안 {va.n_extension_final}건({va.share_extension_final:.1%}). "
    f"영역 순서는 {'유지' if same_order else '일부 변동'}되고 절대 후보 수가 줄어듦",
    f"- **왜 개인 조건에 맞는 연결이 필요한가:** 남는 후보는 영역·지역·신청 경로·자격 조건에 따라 달라짐. "
    f"시군구 한 곳 기준 최종 집 후보 중앙값은 영역별 {home_med.min():.0f}~{home_med.max():.0f}건",
]))


- 조건별 건수는 자동 라벨 중심의 근사값, 이용률·정원·수혜 인원 추정 불가
- 누락 검증은 수요 10개 영역의 청년 대상 후보 중 자동 집 조건이 아닌 나머지 전체를 검토함
- 엄격 S2 청년 범위 밖 후보, SSIS 미등록 사업, ⑪ 주거 단독 후보는 검증 대상이 아님
- 확인 불가 서비스는 최종 상한에만 포함함
- 공식 자료 확인은 일부 중앙 서비스에 한정됨
- 원문 기준 적용 후 정밀도는 사람 판정을 본 뒤 맞춘 보정 결과이며 독립 검증이 아님
- 검수 표본 내 재현율은 비무작위 표본의 값이며 전체 재현율이 아님
- ② 확장분은 자격 조건이 있는 돌봄과 어휘 매칭분, 직접 회복 후보 증가가 아님
- 일상생활 회복의 직접 회복 3건과 공간(①) 0건은 SSIS 등록 자료와 분류 기준에 따른 결과이며 그런 서비스가 없다는 증명이 아님
- 155번의 반복 양육생활비와 217번의 직접 회복은 개별 합의 반영
- 돌봄 범위·정기 수당·용도 미상 축하금·목적성 비용은 정의별 민감도 근거로 사용함
- 조사와 서비스 수집 시점 차이, 비확률 표본 및 검수 범위에 따른 해석 제한

# Appendix. 탐색 회귀

- 매우 필요(4) 여부의 영역별 로지스틱 회귀, 공변량 결측 제외
- 모형 A는 은둔 표본 외출 정도·기간·연령·성별, 모형 B는 은둔·고립 보조 집단·연령·성별
- BH 보정 범위는 모형 A의 외출 정도·기간 항 90개와 모형 B의 집단 항 10개
- 분리·미수렴은 status에 기록하며 기술적 연관으로 해석함


In [ ]:
lg = save_csv(D.fit_logit_appendix(hiki, iso), "appendix_logit_results")
foc = lg[lg["is_focal"]]
fa = foc[foc["model"] == "within_hikikomori"]; fb = foc[foc["model"] == "group_compare"]
show_table(fb[["area_label", "term_label", "odds_ratio", "or_ci95_lo", "or_ci95_hi", "p_value", "p_bh", "n_used", "n_events", "status"]],
           "표 App-1. 모형 B(집단 비교): 은둔 vs 고립 보조 오즈비 (연령·성별 통제)", fmt={c: "{:.3f}" for c in ["odds_ratio", "or_ci95_lo", "or_ci95_hi", "p_value", "p_bh"]} | {"n_used": "{:,.0f}", "n_events": "{:,.0f}"},
           note="모형 A(90항)는 appendix_logit_results.csv. 오즈비는 %p 차이(PART B)와 다른 지표다.")

In [ ]:
md(f"""**관찰.** 적합 {lg.groupby(['model', 'area']).ngroups}개 중 상태가 ok가 아닌 것은 {int((lg.drop_duplicates(['model', 'area'])['status'] != 'ok').sum())}개다. 모형 A의 초점 항 {len(fa)}개 중 BH 보정 p < 0.05는 {int((fa['p_bh'] < 0.05).sum())}개, 모형 B는 {len(fb)}개 중 {int((fb['p_bh'] < 0.05).sum())}개이다.
모형 B 오즈비 범위는 {fb['odds_ratio'].min():.2f} ~ {fb['odds_ratio'].max():.2f}이다.

**해석과 한계.** 탐색적 부록이며 본문 결론·공급 순위에 쓰지 않는다. 비확률 표본이고 은둔·고립 추출 기준과 문항 문구가 달라 집단 항을 효과로 읽을 수 없다. 외출 정도·기간·집단은 서로 얽혀 있고 관찰 연관이다.""")

### Decision Box 10 — 패턴 표시의 경계 (결과를 본 뒤 조정)
- **문제:** '수요 상위인데 공급 순위가 낮은' 영역을 어디까지 검토 대상으로 표시할지. 처음 정한 경계는 주 지표 3위 이내 & 공급 8위 이하였다.
- **선택:** 처음 경계를 유지하되, 보조 지표 3위 이내(`n_cells_demand34_top3_supply_bottom3`)와 주 지표 5위 이내(`n_cells_demand_top5_supply_bottom3`)를 함께 센다. 셋 중 하나라도 있으면 `pattern_note`를 붙인다. 12칸 중 한 칸 이상이면 표시한다.
- **근거:** **결과를 본 뒤 조정했다.** 주 지표 3·4위의 비율 차이가 작고 구간이 겹쳐, 3위 경계만 쓰면 '해당 없음'이라는 결과가 순위 한 칸에 좌우된다.
- **결론에 미친 영향/한계:** 표시는 탐색적 검토 대상일 뿐이며 공급 부족을 뜻하지 않는다. 경계와 공급 순위 8위 기준에는 이론적 근거가 없다.
- **후속 검증:** 사람 검수로 후보 누락 여부를 확인한 뒤 다시 본다.
